# Notebook 12: Cross-Disease Validation — GSE80655 Schizophrenia

**Dataset:** [GSE80655](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE80655) — Ramaker et al. 2017, *Genome Medicine*  
**Title:** "Post-mortem molecular profiling of three psychiatric disorders"  
**Platform:** Illumina HiSeq 2000 (RNA-seq)  
**Samples:** 281 post-mortem brain RNA-seq samples  
- 4 diagnoses: Schizophrenia (SCZ), Bipolar Disorder (BD), Major Depressive Disorder (MDD), Control  
- 3 brain regions: Anterior Cingulate Cortex (ACC), Dorsolateral Prefrontal Cortex (DLPFC), Nucleus Accumbens (NAc)  
- ~24 subjects per diagnosis group  

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0  
**Author:** Rohit Chauhan ([ORCID: 0009-0003-9895-4629](https://orcid.org/0009-0003-9895-4629))  
**Zenodo DOI:** [10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)  

---

## What this notebook does

1. Downloads GSE80655 expression data + metadata from GEO
2. Preprocesses the expression matrix (gene-level, QC, log-transform)
3. Scores samples on **both** 14 schizophrenia and 15 autism pathway sets (dual scoring)
4. Runs pathway-based GMM subtyping on SCZ + Control samples (SCZ pathways)
5. DLPFC region-focused analysis with k-sweep and full validation
6. Benchmarks against NMF, PCA+K-means, gene-level K-means, and random baseline
7. **Cross-disease analysis:** Tests whether autism-derived pathways stratify schizophrenia
8. **Multi-diagnosis pooled clustering:** Tests for trans-diagnostic molecular subtypes
9. **Cross-cohort projection:** Projects SCZ samples into the autism GSE28521 model

### Key Scientific Questions

- Do autism-derived pathways stratify schizophrenia? (cross-pathway ARI)
- Do molecular subtypes cut across diagnostic boundaries? (chi-squared test)
- Does the autism GABA-Collapsed subtype appear in SCZ patients? (cross-cohort projection)
- Which pathways drive SCZ subtype separation? (characterization)
- Are subtypes brain-region-specific or brain-wide? (cross-region ARI)

**Runtime:** ~10-15 minutes on Colab Pro

## 1. Setup & Installation

In [ ]:
# Install pathway-subtyping framework with visualization extras
!pip install -q pathway-subtyping[viz]==0.3.0 GEOparse mygene

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', message='.*Covariance.*')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import urllib.request

# Framework imports
from pathway_subtyping import (
    score_pathways_from_expression,
    ExpressionScoringMethod,
    run_clustering,
    ClusteringAlgorithm,
    select_n_clusters,
    compare_algorithms,
    ValidationGates,
    characterize_subtypes,
    generate_subtype_heatmap,
    generate_gene_heatmap,
    export_characterization,
    run_benchmark_comparison,
    compute_dim_reduction,
    DimReductionMethod,
)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Output directories
DATA_DIR = "./data"
OUTPUT_DIR = "./outputs/gse80655"
DLPFC_DIR = os.path.join(OUTPUT_DIR, "dlpfc")
CROSS_DISEASE_DIR = os.path.join(OUTPUT_DIR, "cross_disease")
for d in [DATA_DIR, OUTPUT_DIR, DLPFC_DIR, CROSS_DISEASE_DIR]:
    os.makedirs(d, exist_ok=True)

print("Setup complete.")

## 2. Download GSE80655 from GEO

GSE80655 is an RNA-seq dataset with 281 post-mortem brain samples across
3 brain regions and 4 psychiatric diagnoses. We use GEOparse for metadata
and download the supplementary expression file directly from GEO FTP.

In [ ]:
import GEOparse

print("Downloading GSE80655 metadata from GEO (this may take a minute)...")
gse = GEOparse.get_GEO(geo="GSE80655", destdir=DATA_DIR, silent=True)
print(f"Downloaded. Platform(s): {list(gse.gpls.keys())}")
print(f"Number of samples (GSMs): {len(gse.gsms)}")

# Inspect first sample to understand metadata structure
first_gsm_name = list(gse.gsms.keys())[0]
first_gsm = gse.gsms[first_gsm_name]
print(f"\nFirst sample: {first_gsm_name}")
print(f"Title: {first_gsm.metadata.get('title', ['?'])[0]}")
print(f"Source: {first_gsm.metadata.get('source_name_ch1', ['?'])[0]}")
print(f"Characteristics: {first_gsm.metadata.get('characteristics_ch1', [])}")

## 3. Extract Sample Metadata

Parse sample characteristics to extract:
- **Diagnosis:** Schizophrenia (SCZ), Bipolar Disorder (BD), Major Depressive Disorder (MDD), Control
- **Brain region:** ACC (anterior cingulate cortex), DLPFC (dorsolateral prefrontal cortex), NAc (nucleus accumbens)

In [ ]:
# Extract metadata from all samples
metadata_rows = []
for gsm_name, gsm in gse.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    char_dict = {}
    for c in chars:
        if ":" in c:
            key, val = c.split(":", 1)
            char_dict[key.strip().lower()] = val.strip()

    title = gsm.metadata.get("title", [""])[0]
    source = gsm.metadata.get("source_name_ch1", [""])[0]

    metadata_rows.append({
        "sample_id": gsm_name,
        "title": title,
        "source": source,
        **char_dict,
    })

metadata = pd.DataFrame(metadata_rows).set_index("sample_id")

# Display available columns and unique values
print("Metadata columns:", list(metadata.columns))
print(f"\nUnique values per column:")
for col in metadata.columns:
    uniq = metadata[col].unique()
    if len(uniq) <= 20:
        print(f"  {col}: {list(uniq)}")
    else:
        print(f"  {col}: {len(uniq)} unique values")

print(f"\nFirst 5 rows:")
print(metadata.head().to_string())

In [ ]:
# Parse diagnosis and brain region from metadata
# Ramaker et al. uses: SCZ, BD/BPD, MDD, Control
# Brain regions: AnCg/ACC, DLPFC, nAcc/NAc

def parse_diagnosis(meta_row):
    """Extract diagnosis from metadata fields."""
    # Check known metadata columns
    for col in ['disease state', 'diagnosis', 'disease status', 'disease',
                'phenotype', 'condition', 'group']:
        if col in meta_row.index and pd.notna(meta_row[col]):
            val = str(meta_row[col]).lower().strip()
            if any(k in val for k in ['schizo', 'scz', 'sz']):
                return 'SCZ'
            elif any(k in val for k in ['bipolar', 'bd', 'bpd']):
                return 'BD'
            elif any(k in val for k in ['major depress', 'mdd']):
                return 'MDD'
            elif any(k in val for k in ['control', 'normal', 'healthy', 'unaffected']):
                return 'Control'
    
    # Fallback: parse from title
    title = str(meta_row.get('title', '')).lower()
    source = str(meta_row.get('source', '')).lower()
    combined = title + ' ' + source
    if any(k in combined for k in ['schizo', 'scz', 'sz']):
        return 'SCZ'
    elif any(k in combined for k in ['bipolar', 'bd', 'bpd']):
        return 'BD'
    elif any(k in combined for k in ['major depress', 'mdd']):
        return 'MDD'
    elif any(k in combined for k in ['control', 'normal', 'healthy']):
        return 'Control'
    
    # Last resort: parse from sample title naming convention
    # Pattern: XXXX_Region_DxCode_SLID (e.g., X1834_AnCg_C_SL31501)
    title_parts = str(meta_row.get('title', '')).split('_')
    if len(title_parts) >= 3:
        code = title_parts[2].upper()
        code_map = {'C': 'Control', 'S': 'SCZ', 'B': 'BD', 'D': 'MDD'}
        if code in code_map:
            return code_map[code]
    
    return 'Unknown'


def parse_brain_region(meta_row):
    """Extract brain region from metadata fields."""
    # Check known metadata columns
    for col in ['tissue', 'brain region', 'region', 'tissue type', 'organ']:
        if col in meta_row.index and pd.notna(meta_row[col]):
            val = str(meta_row[col]).lower().strip()
            if any(k in val for k in ['anterior cingulate', 'ancg', 'acc']):
                return 'ACC'
            elif any(k in val for k in ['dorsolateral', 'dlpfc']):
                return 'DLPFC'
            elif any(k in val for k in ['nucleus accumbens', 'nacc', 'nac']):
                return 'NAc'
    
    # Fallback: parse from title
    title = str(meta_row.get('title', '')).lower()
    source = str(meta_row.get('source', '')).lower()
    combined = title + ' ' + source
    if any(k in combined for k in ['anterior cingulate', 'ancg', 'acc']):
        return 'ACC'
    elif any(k in combined for k in ['dorsolateral', 'dlpfc']):
        return 'DLPFC'
    elif any(k in combined for k in ['nucleus accumbens', 'nacc', 'nac']):
        return 'NAc'
    
    # Last resort: parse from naming convention (XXXX_Region_Dx_SLID)
    title_parts = str(meta_row.get('title', '')).split('_')
    if len(title_parts) >= 2:
        region_code = title_parts[1].lower()
        if 'ancg' in region_code or 'acc' in region_code:
            return 'ACC'
        elif 'dlpfc' in region_code:
            return 'DLPFC'
        elif 'nacc' in region_code or 'nac' in region_code:
            return 'NAc'
    
    return 'Unknown'


metadata['diagnosis'] = metadata.apply(parse_diagnosis, axis=1)
metadata['brain_region'] = metadata.apply(parse_brain_region, axis=1)

print("--- Sample Breakdown ---")
ct = metadata.groupby(['brain_region', 'diagnosis']).size().unstack(fill_value=0)
print(ct)
print(f"\nTotal samples: {len(metadata)}")
print(f"\nDiagnosis counts:")
for dx in ['SCZ', 'BD', 'MDD', 'Control', 'Unknown']:
    n = (metadata['diagnosis'] == dx).sum()
    if n > 0:
        print(f"  {dx}: {n}")
print(f"\nRegion counts:")
for region in ['ACC', 'DLPFC', 'NAc', 'Unknown']:
    n = (metadata['brain_region'] == region).sum()
    if n > 0:
        print(f"  {region}: {n}")

## 4. Build Expression Matrix

Download the processed gene expression data from the GEO supplementary file.
This is the `GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz` file containing
normalized expression values for all 281 samples.

In [ ]:
# Download the supplementary expression file directly from GEO FTP
EXPR_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE80nnn/GSE80655/suppl/GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz"
EXPR_PATH = os.path.join(DATA_DIR, "GSE80655_GeneExpressionData_Updated_3-26-2018.txt.gz")

if not os.path.exists(EXPR_PATH):
    print("Downloading GSE80655 expression data from GEO FTP...")
    urllib.request.urlretrieve(EXPR_URL, EXPR_PATH)
    print(f"Downloaded: {EXPR_PATH}")
else:
    print(f"Using cached: {EXPR_PATH}")

# Read the expression matrix
expression_raw = pd.read_csv(EXPR_PATH, sep="\t", index_col=0, compression="gzip")
print(f"\nRaw expression matrix: {expression_raw.shape[0]} rows x {expression_raw.shape[1]} columns")
print(f"\nColumn names (first 5): {list(expression_raw.columns[:5])}")
print(f"Row index (first 5): {list(expression_raw.index[:5])}")
print(f"\nFirst 3x3 corner:")
print(expression_raw.iloc[:3, :3])

In [ ]:
# Inspect the gene ID format and map to gene symbols if needed
sample_ids = list(expression_raw.index[:10])
print("Sample row IDs (first 10):")
for sid in sample_ids:
    print(f"  {sid}")

# Check if row IDs are Ensembl IDs, gene symbols, or something else
first_id = str(expression_raw.index[0])
if first_id.startswith("ENSG"):
    id_format = "Ensembl"
elif first_id.isdigit():
    id_format = "Entrez"
else:
    # Check if they look like gene symbols (short alphanumeric strings)
    id_format = "Gene Symbol" if len(first_id) < 20 else "Unknown"

print(f"\nDetected ID format: {id_format}")
print(f"Expression value range: [{expression_raw.values.min():.4f}, {expression_raw.values.max():.4f}]")
print(f"Mean value: {expression_raw.values.mean():.4f}")

In [ ]:
# Determine orientation: are rows genes or samples?
# If rows are genes: many rows (~15-20k), few columns (~281)
# If rows are samples: few rows (~281), many columns (~15-20k)

n_rows, n_cols = expression_raw.shape
print(f"Matrix shape: {n_rows} rows x {n_cols} columns")

if n_rows > n_cols:
    print("Rows appear to be genes, columns are samples")
    # Check if column names match GSM IDs or sample titles
    col_matches_gsm = sum(1 for c in expression_raw.columns if str(c).startswith('GSM'))
    print(f"Columns matching GSM pattern: {col_matches_gsm}")
    genes_are_rows = True
else:
    print("Rows appear to be samples, columns are genes")
    genes_are_rows = False

# Build gene expression matrix (samples x genes)
if genes_are_rows:
    expression_df = expression_raw.copy()  # genes x samples
else:
    expression_df = expression_raw.T  # transpose to genes x samples

print(f"\nExpression matrix (genes x samples): {expression_df.shape[0]} genes x {expression_df.shape[1]} samples")

In [ ]:
# Map expression columns to GSM IDs
# Column names may be sample titles, brain bank IDs, or GSM IDs
print("--- Mapping expression columns to GSM IDs ---")
print(f"Expression columns (first 5): {list(expression_df.columns[:5])}")
print(f"Metadata index (first 5): {list(metadata.index[:5])}")

# Check direct match
direct_matches = expression_df.columns.intersection(metadata.index)
if len(direct_matches) > 0:
    print(f"\nDirect GSM matches: {len(direct_matches)} / {len(expression_df.columns)}")
    col_to_gsm = {c: c for c in direct_matches}
else:
    print("No direct GSM matches. Building mapping from sample titles...")
    
    # Build mapping: match expression column names to GSM sample titles
    col_to_gsm = {}
    gsm_titles = {gsm_name: gsm.metadata.get('title', [''])[0]
                  for gsm_name, gsm in gse.gsms.items()}
    
    for col in expression_df.columns:
        col_str = str(col).strip()
        
        # Try exact title match
        for gsm_name, title in gsm_titles.items():
            if col_str == title or col_str in title or title in col_str:
                col_to_gsm[col] = gsm_name
                break
        
        if col not in col_to_gsm:
            # Try matching on brain bank ID (first part of column name)
            col_parts = col_str.replace('-', '_').split('_')
            for gsm_name, title in gsm_titles.items():
                title_parts = title.replace('-', '_').split('_')
                # Match if first part (subject ID) and second part (region) align
                if (len(col_parts) >= 2 and len(title_parts) >= 2 and
                    col_parts[0] == title_parts[0]):
                    col_to_gsm[col] = gsm_name
                    break
    
    print(f"Mapped {len(col_to_gsm)} / {len(expression_df.columns)} columns to GSM IDs")
    
    if len(col_to_gsm) == 0:
        # Last resort: if counts match, try positional alignment
        print("\nAttempting fuzzy matching...")
        for col in expression_df.columns:
            col_lower = str(col).lower().strip()
            for gsm_name, title in gsm_titles.items():
                title_lower = title.lower().strip()
                # Check if column contains any substring of the title
                if any(part in col_lower for part in title_lower.split('_') if len(part) > 3):
                    if gsm_name not in col_to_gsm.values():
                        col_to_gsm[col] = gsm_name
                        break
        print(f"Fuzzy mapped {len(col_to_gsm)} / {len(expression_df.columns)} columns")

# Show a few mapping examples
if col_to_gsm:
    print("\nMapping examples (first 5):")
    for i, (col, gsm) in enumerate(list(col_to_gsm.items())[:5]):
        title = gsm_titles.get(gsm, '?')
        print(f"  '{col}' -> {gsm} (title: '{title}')")

In [ ]:
# Apply column mapping and handle gene IDs

# Rename columns to GSM IDs
if col_to_gsm:
    expression_df = expression_df.rename(columns=col_to_gsm)
    # Keep only mapped columns
    gsm_cols = [c for c in expression_df.columns if str(c).startswith('GSM')]
    if gsm_cols:
        expression_df = expression_df[gsm_cols]
        print(f"Expression matrix after GSM mapping: {expression_df.shape[0]} genes x {expression_df.shape[1]} samples")
    else:
        print("WARNING: No GSM columns after mapping. Keeping original columns.")

# Handle gene ID mapping
if id_format == "Ensembl":
    print("\n--- Mapping Ensembl IDs to gene symbols ---")
    try:
        import mygene
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', '-q', 'mygene'])
        import mygene
    
    mg = mygene.MyGeneInfo()
    ensembl_ids = list(expression_df.index)
    # Strip version numbers (ENSG00000123456.7 -> ENSG00000123456)
    clean_ids = [eid.split('.')[0] for eid in ensembl_ids]
    
    print(f"Querying mygene for {len(clean_ids)} Ensembl IDs...")
    results = mg.querymany(clean_ids, scopes='ensembl.gene', fields='symbol',
                           species='human', returnall=True, verbose=False)
    
    ensembl_to_symbol = {}
    for hit in results['out']:
        if 'symbol' in hit and 'query' in hit:
            ensembl_to_symbol[hit['query']] = hit['symbol']
    
    # Map original IDs (with version) to symbols
    id_to_symbol = {}
    for orig_id, clean_id in zip(ensembl_ids, clean_ids):
        if clean_id in ensembl_to_symbol:
            id_to_symbol[orig_id] = ensembl_to_symbol[clean_id]
    
    mapped_mask = expression_df.index.isin(id_to_symbol.keys())
    expression_df = expression_df.loc[mapped_mask]
    expression_df.index = [id_to_symbol[eid] for eid in expression_df.index]
    print(f"Mapped {mapped_mask.sum()} / {len(ensembl_ids)} to gene symbols")

elif id_format == "Entrez":
    print("\n--- Mapping Entrez IDs to gene symbols ---")
    try:
        import mygene
    except ImportError:
        import subprocess
        subprocess.check_call(['pip', 'install', '-q', 'mygene'])
        import mygene
    
    mg = mygene.MyGeneInfo()
    entrez_ids = [str(eid) for eid in expression_df.index]
    
    print(f"Querying mygene for {len(entrez_ids)} Entrez IDs...")
    results = mg.querymany(entrez_ids, scopes='entrezgene', fields='symbol',
                           species='human', returnall=True, verbose=False)
    
    entrez_to_symbol = {}
    for hit in results['out']:
        if 'symbol' in hit and 'query' in hit:
            entrez_to_symbol[hit['query']] = hit['symbol']
    
    mapped_mask = expression_df.index.astype(str).isin(entrez_to_symbol.keys())
    expression_df = expression_df.loc[mapped_mask]
    expression_df.index = [entrez_to_symbol[str(eid)] for eid in expression_df.index]
    print(f"Mapped {mapped_mask.sum()} / {len(entrez_ids)} to gene symbols")

else:
    print(f"\nGene IDs appear to be gene symbols already ({id_format})")

# Collapse duplicate gene symbols by taking the mean
n_before = len(expression_df)
expression_df = expression_df.groupby(expression_df.index).mean()
n_after = len(expression_df)
if n_before != n_after:
    print(f"Collapsed {n_before} -> {n_after} unique genes (mean of duplicates)")

print(f"\nGene-level expression: {expression_df.shape[0]} genes x {expression_df.shape[1]} samples")

In [ ]:
# Transpose to samples x genes and apply QC
gene_expression = expression_df.T
print(f"Transposed: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")

# Ensure numeric
gene_expression = gene_expression.apply(pd.to_numeric, errors='coerce')

# Check log-transform status
max_val = gene_expression.max().max()
min_val = gene_expression.min().min()
print(f"\nExpression range: [{min_val:.4f}, {max_val:.4f}]")
print(f"Mean: {gene_expression.mean().mean():.4f}")

if max_val > 30:
    print("Data appears to be in raw/linear scale — applying log2(x+1) transform")
    gene_expression = np.log2(gene_expression.clip(lower=0) + 1)
    print(f"After log2: range [{gene_expression.min().min():.4f}, {gene_expression.max().max():.4f}]")
else:
    print("Data appears already log-transformed — no additional transform")

# Drop genes with zero variance
gene_var = gene_expression.var()
n_zero_var = (gene_var == 0).sum()
if n_zero_var > 0:
    gene_expression = gene_expression.loc[:, gene_var > 0]
    print(f"Dropped {n_zero_var} zero-variance genes. Remaining: {gene_expression.shape[1]}")

# Fill NaN with column medians
n_nan = gene_expression.isna().sum().sum()
if n_nan > 0:
    gene_expression = gene_expression.fillna(gene_expression.median())
    print(f"Filled {n_nan} NaN values with column medians")

# Align with metadata
common_samples = gene_expression.index.intersection(metadata.index)
if len(common_samples) > 0:
    gene_expression = gene_expression.loc[common_samples]
    metadata = metadata.loc[common_samples]
    print(f"\nAligned: {len(common_samples)} samples in both expression and metadata")
else:
    print(f"\nWARNING: No overlapping sample IDs between expression and metadata")
    print(f"  Expression IDs: {list(gene_expression.index[:3])}")
    print(f"  Metadata IDs: {list(metadata.index[:3])}")
    # If same count, align positionally
    if len(gene_expression) == len(metadata):
        print("  Same count — aligning positionally")
        gene_expression.index = metadata.index
        common_samples = metadata.index

print(f"\n--- Final Expression Matrix ---")
print(f"Shape: {gene_expression.shape[0]} samples x {gene_expression.shape[1]} genes")
print(f"Expression range: [{gene_expression.min().min():.4f}, {gene_expression.max().max():.4f}]")
print(f"Mean: {gene_expression.mean().mean():.4f}")
print(f"\n--- Sample Breakdown (aligned) ---")
print(metadata.groupby(['brain_region', 'diagnosis']).size().unstack(fill_value=0))

## 5. Load Pathway Gene Sets

Load **both** the schizophrenia (14 pathways) and autism (15 pathways) gene sets.
We'll use SCZ pathways for primary subtyping and ASD pathways for cross-disease analysis.

The two pathway sets share 8 pathways by name (with partially overlapping gene lists):
SYNAPTIC_TRANSMISSION, CHROMATIN_REMODELING, WNT_SIGNALING, MTOR_SIGNALING,
GABA_SIGNALING, GLUTAMATE_SIGNALING, CALCIUM_SIGNALING, CIRCADIAN_RHYTHM.

In [ ]:
# Download both GMT files from the framework repository
GMT_BASE = "https://raw.githubusercontent.com/topmist-admin/pathway-subtyping-framework/main/data/pathways"

def load_gmt(url, local_path):
    """Download and parse a GMT file."""
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(url, local_path)
        print(f"Downloaded: {os.path.basename(local_path)}")
    
    pathways = {}
    with open(local_path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                pathways[parts[0]] = parts[2:]
    return pathways


# Load schizophrenia pathways
scz_pathways = load_gmt(
    f"{GMT_BASE}/schizophrenia_pathways.gmt",
    os.path.join(DATA_DIR, "schizophrenia_pathways.gmt")
)

# Load autism pathways
asd_pathways = load_gmt(
    f"{GMT_BASE}/autism_pathways.gmt",
    os.path.join(DATA_DIR, "autism_pathways.gmt")
)

print(f"=== Schizophrenia Pathways ({len(scz_pathways)}) ===")
scz_total_genes = set()
for name, genes in scz_pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    scz_total_genes.update(genes)
    print(f"  {name}: {len(genes)} genes ({available} in expression data)")

print(f"\n=== Autism Pathways ({len(asd_pathways)}) ===")
asd_total_genes = set()
for name, genes in asd_pathways.items():
    available = len(set(genes) & set(gene_expression.columns))
    asd_total_genes.update(genes)
    print(f"  {name}: {len(genes)} genes ({available} in expression data)")

# Identify shared pathways
shared_pathway_names = sorted(set(scz_pathways.keys()) & set(asd_pathways.keys()))

print(f"\n=== Pathway Overlap ===")
print(f"SCZ-only pathways: {sorted(set(scz_pathways.keys()) - set(asd_pathways.keys()))}")
print(f"ASD-only pathways: {sorted(set(asd_pathways.keys()) - set(scz_pathways.keys()))}")
print(f"Shared pathways ({len(shared_pathway_names)}): {shared_pathway_names}")

# Gene overlap within shared pathways
print(f"\n=== Gene Overlap in Shared Pathways ===")
for pw_name in shared_pathway_names:
    scz_genes = set(scz_pathways[pw_name])
    asd_genes = set(asd_pathways[pw_name])
    overlap = scz_genes & asd_genes
    union = scz_genes | asd_genes
    jaccard = len(overlap) / len(union) if union else 0
    print(f"  {pw_name}: {len(scz_genes)} SCZ genes, {len(asd_genes)} ASD genes, "
          f"{len(overlap)} shared (Jaccard={jaccard:.2f})")

print(f"\n=== Gene Coverage Summary ===")
print(f"Total unique SCZ pathway genes: {len(scz_total_genes)} ({len(scz_total_genes & set(gene_expression.columns))} in data)")
print(f"Total unique ASD pathway genes: {len(asd_total_genes)} ({len(asd_total_genes & set(gene_expression.columns))} in data)")
all_pathway_genes = scz_total_genes | asd_total_genes
print(f"Combined unique genes: {len(all_pathway_genes)} ({len(all_pathway_genes & set(gene_expression.columns))} in data)")

In [ ]:
# Save processed data for downstream sections
gene_expression.to_csv(os.path.join(OUTPUT_DIR, "gene_expression_processed.csv"))
metadata.to_csv(os.path.join(OUTPUT_DIR, "sample_metadata.csv"))

print("\n" + "=" * 60)
print("PART 1 COMPLETE: Data Acquisition & Preprocessing")
print("=" * 60)
print(f"\nDataset: GSE80655 (Ramaker et al. 2017, Genome Medicine)")
print(f"Samples: {len(gene_expression)}")
print(f"  SCZ: {(metadata['diagnosis'] == 'SCZ').sum()}")
print(f"  BD:  {(metadata['diagnosis'] == 'BD').sum()}")
print(f"  MDD: {(metadata['diagnosis'] == 'MDD').sum()}")
print(f"  CTL: {(metadata['diagnosis'] == 'Control').sum()}")
print(f"Regions: {list(metadata['brain_region'].unique())}")
print(f"Genes: {gene_expression.shape[1]}")
print(f"SCZ pathways: {len(scz_pathways)}")
print(f"ASD pathways: {len(asd_pathways)}")
print(f"Shared pathways: {len(shared_pathway_names)}")
print(f"\nOutputs saved to: {OUTPUT_DIR}/")
print(f"  - gene_expression_processed.csv")
print(f"  - sample_metadata.csv")

## 6. Dual Pathway Scoring

Score all samples on **both** pathway sets using ssGSEA:
- **SCZ pathways** (14) — for primary schizophrenia subtyping
- **ASD pathways** (15) — for cross-disease comparison in later sections

This produces two pathway score matrices that will be used throughout the notebook.

In [ ]:
# Score all samples with SCZ pathways (14 pathways)
print("Scoring with schizophrenia pathways...")
scz_scoring = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=scz_pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores_scz = scz_scoring.pathway_scores
print(f"\n--- SCZ Scoring Report ---")
print(scz_scoring.format_report())
print(f"SCZ pathway score matrix: {pathway_scores_scz.shape}")

# Score all samples with ASD pathways (15 pathways)
print("\nScoring with autism pathways...")
asd_scoring = score_pathways_from_expression(
    gene_expression=gene_expression,
    pathways=asd_pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
    show_progress=True,
)

pathway_scores_asd = asd_scoring.pathway_scores
print(f"\n--- ASD Scoring Report ---")
print(asd_scoring.format_report())
print(f"ASD pathway score matrix: {pathway_scores_asd.shape}")

# Save both score matrices
pathway_scores_scz.to_csv(os.path.join(OUTPUT_DIR, "pathway_scores_scz.csv"))
pathway_scores_asd.to_csv(os.path.join(OUTPUT_DIR, "pathway_scores_asd.csv"))
print(f"\nSaved: pathway_scores_scz.csv ({pathway_scores_scz.shape})")
print(f"Saved: pathway_scores_asd.csv ({pathway_scores_asd.shape})")

In [ ]:
# Visualize SCZ pathway score distributions by diagnosis
scores_with_meta = pathway_scores_scz.copy()
scores_with_meta["diagnosis"] = metadata.loc[pathway_scores_scz.index, "diagnosis"]
scores_with_meta["brain_region"] = metadata.loc[pathway_scores_scz.index, "brain_region"]

n_pathways = pathway_scores_scz.shape[1]
n_cols = 5
n_rows = (n_pathways + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes_flat = axes.flatten()

dx_colors = {"SCZ": "tomato", "BD": "orange", "MDD": "mediumpurple", "Control": "steelblue"}

for i, pathway in enumerate(pathway_scores_scz.columns):
    ax = axes_flat[i]
    for dx in ["SCZ", "Control", "BD", "MDD"]:
        subset = scores_with_meta[scores_with_meta["diagnosis"] == dx][pathway]
        if len(subset) > 0:
            ax.hist(subset, alpha=0.5, label=dx, color=dx_colors.get(dx, "gray"), bins=12)
    ax.set_title(pathway.replace("_", "\n"), fontsize=8)
    ax.set_xlabel("")
    if i == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for j in range(n_pathways, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("SCZ Pathway Score Distributions by Diagnosis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pathway_distributions_scz.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. SCZ + Control Subtyping

Subset to **schizophrenia and control** samples only for primary subtyping analysis.
Use BIC to select optimal k, then run GMM clustering on SCZ pathway scores.

We cap the k-range based on sample count to avoid overfitting with small subgroups.

In [ ]:
# Subset to SCZ + Control samples only
scz_ctl_mask = metadata["diagnosis"].isin(["SCZ", "Control"])
scz_ctl_scores = pathway_scores_scz.loc[scz_ctl_mask]
scz_ctl_expression = gene_expression.loc[scz_ctl_mask]
scz_ctl_meta = metadata.loc[scz_ctl_mask].copy()

n_scz_ctl = len(scz_ctl_scores)
n_scz = (scz_ctl_meta["diagnosis"] == "SCZ").sum()
n_ctl = (scz_ctl_meta["diagnosis"] == "Control").sum()

print(f"SCZ + Control subset: {n_scz_ctl} samples ({n_scz} SCZ, {n_ctl} Control)")
print(f"Pathway scores: {scz_ctl_scores.shape}")

# Cap k-range for sample size (at least 8 samples per cluster)
max_k = min(7, n_scz_ctl // 8)
max_k = max(max_k, 2)  # at least test k=2
k_range = list(range(2, max_k + 1))
print(f"\nk-range for BIC selection: {k_range} (capped at n/{8} = {n_scz_ctl // 8})")

# Select optimal number of clusters using BIC
selection = select_n_clusters(
    data=scz_ctl_scores.values,
    k_range=k_range,
    method="bic",
    seed=SEED,
)

optimal_k = selection.optimal_k
print(f"\nOptimal k (BIC): {optimal_k}")
print(f"BIC values: {selection.bic_values}")
print(f"Silhouette values: {selection.silhouette_values}")

# Plot BIC and Silhouette curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ks = sorted(selection.bic_values.keys())
ax1.plot(ks, [selection.bic_values[k] for k in ks], "bo-", linewidth=2)
ax1.axvline(x=optimal_k, color="red", linestyle="--", label=f"Optimal k={optimal_k}")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("BIC (lower is better)")
ax1.set_title("Model Selection: BIC")
ax1.legend()

ax2.plot(ks, [selection.silhouette_values[k] for k in ks], "go-", linewidth=2)
ax2.axvline(x=optimal_k, color="red", linestyle="--", label=f"Optimal k={optimal_k}")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score (higher is better)")
ax2.set_title("Model Selection: Silhouette")
ax2.legend()

plt.suptitle(f"GSE80655 SCZ+CTL: Optimal Cluster Selection (n={n_scz_ctl})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model_selection.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Run GMM clustering at optimal k
clustering = run_clustering(
    data=scz_ctl_scores.values,
    n_clusters=optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)

labels = clustering.labels

print(f"--- GMM Clustering Results (SCZ+CTL) ---")
print(f"k = {clustering.n_clusters}")
print(f"Silhouette: {clustering.silhouette:.4f}")
print(f"Calinski-Harabasz: {clustering.calinski_harabasz:.2f}")
print(f"Davies-Bouldin: {clustering.davies_bouldin:.4f}")
if clustering.bic is not None:
    print(f"BIC: {clustering.bic:.2f}")
print(f"Converged: {clustering.converged}")

# Subtype sizes
print(f"\nSubtype sizes:")
for i in range(optimal_k):
    count = int((labels == i).sum())
    print(f"  Subtype {i}: {count} samples ({count / len(labels) * 100:.1f}%)")

# Cross-tabulate with diagnosis and brain region
scz_ctl_meta["subtype"] = labels

print(f"\n--- Subtype x Diagnosis ---")
ct_dx = pd.crosstab(scz_ctl_meta["subtype"], scz_ctl_meta["diagnosis"], margins=True)
print(ct_dx)

print(f"\n--- Subtype x Brain Region ---")
ct_region = pd.crosstab(scz_ctl_meta["subtype"], scz_ctl_meta["brain_region"], margins=True)
print(ct_region)

print(f"\n--- Subtype x Diagnosis x Region ---")
ct_full = pd.crosstab([scz_ctl_meta["subtype"], scz_ctl_meta["brain_region"]],
                       scz_ctl_meta["diagnosis"])
print(ct_full)

In [ ]:
# PCA scatter plots: subtype, diagnosis, and brain region
embedding, pca_meta = compute_dim_reduction(
    pathway_scores=scz_ctl_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Color by subtype
scatter_colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    axes[0].scatter(embedding[mask, 0], embedding[mask, 1], c=[scatter_colors[i]],
                    label=f"Subtype {i} (n={int(mask.sum())})", s=60, alpha=0.7,
                    edgecolors="k", linewidth=0.5)
axes[0].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0] * 100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1] * 100:.1f}%)")
axes[0].set_title("Colored by Molecular Subtype")
axes[0].legend(fontsize=8)

# Plot 2: Color by diagnosis
dx_colors = {"SCZ": "tomato", "Control": "steelblue"}
for dx, color in dx_colors.items():
    mask = scz_ctl_meta["diagnosis"].values == dx
    axes[1].scatter(embedding[mask, 0], embedding[mask, 1], c=color,
                    label=dx, s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[1].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0] * 100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1] * 100:.1f}%)")
axes[1].set_title("Colored by Diagnosis")
axes[1].legend(fontsize=8)

# Plot 3: Color by brain region
region_colors = {"ACC": "#2ecc71", "DLPFC": "#e74c3c", "NAc": "#3498db"}
for region, color in region_colors.items():
    mask = scz_ctl_meta["brain_region"].values == region
    if mask.any():
        axes[2].scatter(embedding[mask, 0], embedding[mask, 1], c=color,
                        label=region, s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[2].set_xlabel(f"PC1 ({pca_meta['explained_variance_ratio'][0] * 100:.1f}%)")
axes[2].set_ylabel(f"PC2 ({pca_meta['explained_variance_ratio'][1] * 100:.1f}%)")
axes[2].set_title("Colored by Brain Region")
axes[2].legend(fontsize=8)

plt.suptitle(f"GSE80655 SCZ+CTL: Pathway-Based Molecular Subtypes (k={optimal_k}, GMM)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_scatter_trio.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Validation Gates & Characterization

Run the framework's validation gates on SCZ+CTL subtypes:
1. **Negative Control 1 (Label Shuffle):** Shuffled labels should NOT be recoverable
2. **Negative Control 2 (Random Gene Sets):** Random pathways should NOT reproduce the clusters
3. **Stability (Bootstrap):** Clusters should survive resampling

Then characterize each subtype by enriched pathways and top contributing genes.

In [ ]:
# Run validation gates on SCZ+CTL subtypes
gates = ValidationGates(
    seed=SEED,
    n_permutations=200,
    n_bootstrap=100,
    stability_threshold=0.8,
    null_ari_max=0.15,
    show_progress=True,
)

print("Running validation gates (this may take 1-2 minutes)...")
val_result = gates.run_all(
    pathway_scores=scz_ctl_scores,
    cluster_labels=labels,
    pathways=scz_pathways,
    gene_burdens=scz_ctl_expression,
    n_clusters=optimal_k,
    gmm_seed=SEED,
)

print("\n" + "=" * 60)
print("VALIDATION GATES RESULTS (SCZ+CTL)")
print("=" * 60)
print(f"\nAll gates passed: {'YES' if val_result.all_passed else 'NO'}")
print()
for gate in val_result.results:
    status = "PASS" if gate.passed else "FAIL"
    print(f"  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} "
          f"(threshold: {gate.comparison} {gate.threshold:.4f})")

In [ ]:
# Characterize subtypes: enriched pathways and top genes
char_result = characterize_subtypes(
    pathway_scores=scz_ctl_scores,
    cluster_labels=labels,
    gene_burdens=scz_ctl_expression,
    pathways=scz_pathways,
    fdr_alpha=0.05,
    top_n_genes=20,
    seed=SEED,
)

print(char_result.format_report())

In [ ]:
# Generate pathway heatmap
fig_heatmap = generate_subtype_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, "subtype_heatmap.png"),
    figsize=(14, 8),
)
plt.show()

# Generate gene contribution heatmap
fig_genes = generate_gene_heatmap(
    char_result,
    output_path=os.path.join(OUTPUT_DIR, "gene_heatmap.png"),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

# Export characterization data
export_files = export_characterization(
    char_result,
    output_dir=OUTPUT_DIR,
    formats=["csv"],
)
print("Exported characterization files:")
for f in export_files:
    print(f"  {f}")

In [ ]:
# Save SCZ+CTL subtyping results
scz_ctl_meta.to_csv(os.path.join(OUTPUT_DIR, "sample_metadata_with_subtypes.csv"))

scz_ctl_summary = {
    "analysis": "scz_ctl_subtyping",
    "dataset": "GSE80655",
    "citation": "Ramaker et al. 2017, Genome Medicine",
    "n_samples": int(n_scz_ctl),
    "n_scz": int(n_scz),
    "n_control": int(n_ctl),
    "n_genes": int(gene_expression.shape[1]),
    "n_pathways_scored": int(scz_scoring.n_pathways_scored),
    "scoring_method": "ssGSEA",
    "optimal_k": int(optimal_k),
    "k_range_tested": k_range,
    "clustering_algorithm": "GMM",
    "silhouette": float(clustering.silhouette),
    "calinski_harabasz": float(clustering.calinski_harabasz),
    "davies_bouldin": float(clustering.davies_bouldin),
    "validation_all_passed": bool(val_result.all_passed),
    "validation_gates": [
        {"name": str(g.name), "passed": bool(g.passed), "metric": str(g.metric_name),
         "value": float(g.metric_value), "threshold": float(g.threshold)}
        for g in val_result.results
    ],
    "subtype_sizes": {str(i): int((labels == i).sum()) for i in range(optimal_k)},
    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(scz_ctl_summary, f, indent=2)

val_status = "ALL PASSED" if val_result.all_passed else "SOME FAILED"

print("\n" + "=" * 60)
print("PART 2 COMPLETE: Dual Scoring & SCZ+CTL Subtyping")
print("=" * 60)
print(f"\nSCZ pathway scores: {pathway_scores_scz.shape}")
print(f"ASD pathway scores: {pathway_scores_asd.shape}")
print(f"SCZ+CTL samples: {n_scz_ctl} ({n_scz} SCZ, {n_ctl} Control)")
print(f"Optimal k: {optimal_k}")
print(f"Silhouette: {clustering.silhouette:.4f}")
print(f"Validation: {val_status}")
print(f"\nOutputs saved to: {OUTPUT_DIR}/")
print(f"  - pathway_scores_scz.csv, pathway_scores_asd.csv")
print(f"  - sample_metadata_with_subtypes.csv")
print(f"  - results_summary.json")
print(f"  - model_selection.png, pca_scatter_trio.png")
print(f"  - subtype_heatmap.png, gene_heatmap.png")
print(f"  - pathway_distributions_scz.png")

## 9. Benchmark Comparison

Compare pathway-based GMM subtyping against standard approaches:
- **NMF:** Non-negative Matrix Factorization on raw expression
- **PCA + K-means:** PCA dimensionality reduction then K-means
- **Gene-level K-means:** Direct clustering on expression values
- **Random baseline:** Random cluster assignments

This demonstrates that pathway-based scoring captures biologically meaningful
structure that raw expression methods may miss.

In [ ]:
# Run benchmark comparison on SCZ+CTL subset
print("Running benchmark comparison (SCZ+CTL)...")
bench_result = run_benchmark_comparison(
    gene_burdens=scz_ctl_expression,
    pathway_scores=scz_ctl_scores,
    pathways=scz_pathways,
    n_clusters=optimal_k,
    seed=SEED,
)

print("\n" + bench_result.format_report())

In [ ]:
# Visualize benchmark results
methods = list(bench_result.method_results.keys())
silhouettes = [bench_result.method_results[m].silhouette for m in methods]
runtimes = [bench_result.method_results[m].runtime_seconds for m in methods]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Silhouette comparison
colors = ["#2ecc71" if m == bench_result.best_method else "#3498db" for m in methods]
bars = ax1.barh(methods, silhouettes, color=colors)
ax1.set_xlabel("Silhouette Score (higher is better)")
ax1.set_title("Clustering Quality: Method Comparison")
for bar, val in zip(bars, silhouettes):
    ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=10)

# Runtime comparison
ax2.barh(methods, runtimes, color="#9b59b6")
ax2.set_xlabel("Runtime (seconds)")
ax2.set_title("Computational Cost")
for bar, val in zip(ax2.patches, runtimes):
    ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.2f}s", va="center", fontsize=10)

plt.suptitle(f"GSE80655 SCZ+CTL: Benchmark Comparison (k={optimal_k})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "benchmark_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. Algorithm Comparison

Compare clustering algorithms (GMM, K-means, Spectral, Agglomerative) on
the same SCZ pathway scores. This tests whether the subtype structure is
robust to the choice of clustering method.

In [ ]:
# Compare all clustering algorithms on SCZ+CTL pathway scores
algo_comparison = compare_algorithms(
    data=scz_ctl_scores.values,
    n_clusters=optimal_k,
    seed=SEED,
)

print(f"Most stable algorithm: {algo_comparison.most_stable_algorithm}")

print(f"\nPairwise ARI (inter-algorithm agreement):")
for pair, ari in algo_comparison.pairwise_ari.items():
    print(f"  {pair}: {ari:.4f}")

print(f"\nPer-algorithm metrics:")
for algo, res in algo_comparison.results.items():
    print(f"  {algo}: silhouette={res.silhouette:.4f}, "
          f"CH={res.calinski_harabasz:.1f}, DB={res.davies_bouldin:.4f}")

## 11. DLPFC Region-Focused Analysis

The dorsolateral prefrontal cortex (DLPFC) is the brain region most consistently
implicated in schizophrenia cognitive deficits. We run a focused analysis:

1. Subset to DLPFC samples only
2. Sweep k=2,3,4 with full validation per k
3. Pick the best k based on validation gates + silhouette
4. Characterize DLPFC-specific subtypes
5. Benchmark on DLPFC subset
6. Compare subtype structure across all 3 brain regions

In [ ]:
# ── 11a. Subset to DLPFC ────────────────────────────────────────────────────

dlpfc_mask = metadata.loc[pathway_scores_scz.index, "brain_region"] == "DLPFC"
dlpfc_scores = pathway_scores_scz.loc[dlpfc_mask]
dlpfc_expression = gene_expression.loc[dlpfc_mask]
dlpfc_meta = metadata.loc[dlpfc_scores.index].copy()

# Further subset to SCZ + Control for primary analysis
dlpfc_scz_ctl_mask = dlpfc_meta["diagnosis"].isin(["SCZ", "Control"])
dlpfc_scz_ctl_scores = dlpfc_scores.loc[dlpfc_scz_ctl_mask]
dlpfc_scz_ctl_expr = dlpfc_expression.loc[dlpfc_scz_ctl_mask]
dlpfc_scz_ctl_meta = dlpfc_meta.loc[dlpfc_scz_ctl_mask].copy()

n_dlpfc = len(dlpfc_scz_ctl_scores)
n_dlpfc_scz = (dlpfc_scz_ctl_meta["diagnosis"] == "SCZ").sum()
n_dlpfc_ctl = (dlpfc_scz_ctl_meta["diagnosis"] == "Control").sum()

print("=" * 60)
print("DLPFC REGION-FOCUSED ANALYSIS")
print("=" * 60)
print(f"\nAll DLPFC samples: {len(dlpfc_scores)}")
print(f"  SCZ:     {(dlpfc_meta['diagnosis'] == 'SCZ').sum()}")
print(f"  BD:      {(dlpfc_meta['diagnosis'] == 'BD').sum()}")
print(f"  MDD:     {(dlpfc_meta['diagnosis'] == 'MDD').sum()}")
print(f"  Control: {(dlpfc_meta['diagnosis'] == 'Control').sum()}")
print(f"\nSCZ+CTL subset: {n_dlpfc} ({n_dlpfc_scz} SCZ, {n_dlpfc_ctl} Control)")
print(f"Pathway scores: {dlpfc_scz_ctl_scores.shape}")

# ── 11b. k-sweep (k=2,3,4) with validation + characterization ──────────────

# Cap k-range for DLPFC sample size
dlpfc_max_k = min(4, n_dlpfc // 8)
dlpfc_max_k = max(dlpfc_max_k, 2)
dlpfc_k_values = list(range(2, dlpfc_max_k + 1))
print(f"\nk values to sweep: {dlpfc_k_values} (capped at n/{8} = {n_dlpfc // 8})")

dlpfc_all_results = {}

for k in dlpfc_k_values:
    print(f"\n{'=' * 60}")
    print(f"DLPFC — k={k}")
    print(f"{'=' * 60}")
    
    # Cluster
    dlpfc_clustering_k = run_clustering(
        data=dlpfc_scz_ctl_scores.values,
        n_clusters=k,
        algorithm=ClusteringAlgorithm.GMM,
        seed=SEED,
    )
    dlpfc_labels_k = dlpfc_clustering_k.labels
    
    print(f"\nSilhouette: {dlpfc_clustering_k.silhouette:.4f}")
    print(f"Calinski-Harabasz: {dlpfc_clustering_k.calinski_harabasz:.2f}")
    print(f"Davies-Bouldin: {dlpfc_clustering_k.davies_bouldin:.4f}")
    
    # Cross-tab with diagnosis
    dlpfc_sub_meta = dlpfc_scz_ctl_meta.copy()
    dlpfc_sub_meta["subtype"] = dlpfc_labels_k
    ct = pd.crosstab(dlpfc_sub_meta["subtype"], dlpfc_sub_meta["diagnosis"], margins=True)
    print(f"\nSubtype x Diagnosis:")
    print(ct)
    
    # Validation gates
    dlpfc_gates = ValidationGates(
        seed=SEED,
        n_permutations=200,
        n_bootstrap=100,
        stability_threshold=0.8,
        null_ari_max=0.15,
        show_progress=False,
    )
    
    dlpfc_val_k = dlpfc_gates.run_all(
        pathway_scores=dlpfc_scz_ctl_scores,
        cluster_labels=dlpfc_labels_k,
        pathways=scz_pathways,
        gene_burdens=dlpfc_scz_ctl_expr,
        n_clusters=k,
        gmm_seed=SEED,
    )
    
    n_passed = sum(1 for g in dlpfc_val_k.results if g.passed)
    print(f"\nValidation Gates:")
    for gate in dlpfc_val_k.results:
        status = "PASS" if gate.passed else "FAIL"
        print(f"  [{status}] {gate.name}: {gate.metric_name} = {gate.metric_value:.4f} "
              f"(threshold: {gate.comparison} {gate.threshold:.4f})")
    print(f"\nGates passed: {n_passed}/{len(dlpfc_val_k.results)} — All passed: {dlpfc_val_k.all_passed}")
    
    # Characterize
    dlpfc_char_k = characterize_subtypes(
        pathway_scores=dlpfc_scz_ctl_scores,
        cluster_labels=dlpfc_labels_k,
        gene_burdens=dlpfc_scz_ctl_expr,
        pathways=scz_pathways,
        fdr_alpha=0.05,
        top_n_genes=15,
        seed=SEED,
    )
    
    dlpfc_all_results[k] = {
        "clustering": dlpfc_clustering_k,
        "labels": dlpfc_labels_k,
        "validation": dlpfc_val_k,
        "characterization": dlpfc_char_k,
        "n_gates_passed": n_passed,
        "silhouette": float(dlpfc_clustering_k.silhouette),
        "meta": dlpfc_sub_meta,
    }

In [ ]:
# ── 11c. Compare k values — pick the best ────────────────────────────────────

print("=" * 60)
print("DLPFC: k COMPARISON SUMMARY")
print("=" * 60)
print(f"\n{'k':<4} {'Silhouette':<12} {'Gates Passed':<14} {'All Passed':<12}")
print("-" * 42)
for k in dlpfc_k_values:
    r = dlpfc_all_results[k]
    n_total = len(r['validation'].results)
    all_pass = 'YES' if r['validation'].all_passed else 'NO'
    print(f"{k:<4} {r['silhouette']:<12.4f} {r['n_gates_passed']}/{n_total:<12} {all_pass}")

# Select best k: prioritize validation gates passed, then silhouette
best_dlpfc_k = max(dlpfc_all_results.keys(),
                   key=lambda k: (dlpfc_all_results[k]["n_gates_passed"],
                                  dlpfc_all_results[k]["silhouette"]))

print(f"\nBest k for DLPFC: {best_dlpfc_k}")
print(f"  Silhouette: {dlpfc_all_results[best_dlpfc_k]['silhouette']:.4f}")
print(f"  Gates passed: {dlpfc_all_results[best_dlpfc_k]['n_gates_passed']}")

# Compare to full-dataset analysis
full_gates_passed = sum(1 for g in val_result.results if g.passed)
print(f"\n--- Comparison: Full Dataset vs DLPFC ---")
print(f"  Full SCZ+CTL: k={optimal_k}, silhouette={clustering.silhouette:.4f}, "
      f"gates={full_gates_passed}/{len(val_result.results)}")
print(f"  DLPFC only:   k={best_dlpfc_k}, silhouette={dlpfc_all_results[best_dlpfc_k]['silhouette']:.4f}, "
      f"gates={dlpfc_all_results[best_dlpfc_k]['n_gates_passed']}/"
      f"{len(dlpfc_all_results[best_dlpfc_k]['validation'].results)}")

In [ ]:
# ── 11d. Visualize best DLPFC k ──────────────────────────────────────────────

best_dlpfc = dlpfc_all_results[best_dlpfc_k]
best_dlpfc_labels = best_dlpfc["labels"]

# PCA scatter for DLPFC subtypes
dlpfc_embedding, dlpfc_pca_meta = compute_dim_reduction(
    pathway_scores=dlpfc_scz_ctl_scores,
    method=DimReductionMethod.PCA,
    n_components=2,
    seed=SEED,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Color by subtype
scatter_colors = plt.cm.Set2(np.linspace(0, 1, best_dlpfc_k))
for i in range(best_dlpfc_k):
    mask = best_dlpfc_labels == i
    ax1.scatter(dlpfc_embedding[mask, 0], dlpfc_embedding[mask, 1],
               c=[scatter_colors[i]], label=f"Subtype {i} (n={int(mask.sum())})",
               s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
ax1.set_xlabel(f"PC1 ({dlpfc_pca_meta['explained_variance_ratio'][0] * 100:.1f}%)")
ax1.set_ylabel(f"PC2 ({dlpfc_pca_meta['explained_variance_ratio'][1] * 100:.1f}%)")
ax1.set_title("DLPFC: Colored by Subtype")
ax1.legend(fontsize=9)

# Color by diagnosis
dx_colors = {"SCZ": "tomato", "Control": "steelblue"}
for dx, color in dx_colors.items():
    mask = dlpfc_scz_ctl_meta["diagnosis"].values == dx
    ax2.scatter(dlpfc_embedding[mask, 0], dlpfc_embedding[mask, 1], c=color,
               label=dx, s=80, alpha=0.8, edgecolors="k", linewidth=0.5)
ax2.set_xlabel(f"PC1 ({dlpfc_pca_meta['explained_variance_ratio'][0] * 100:.1f}%)")
ax2.set_ylabel(f"PC2 ({dlpfc_pca_meta['explained_variance_ratio'][1] * 100:.1f}%)")
ax2.set_title("DLPFC: Colored by Diagnosis")
ax2.legend(fontsize=9)

plt.suptitle(f"GSE80655 DLPFC SCZ+CTL: Molecular Subtypes (k={best_dlpfc_k})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(DLPFC_DIR, "dlpfc_pca_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

# Heatmaps for best DLPFC k
fig_pw = generate_subtype_heatmap(
    best_dlpfc["characterization"],
    output_path=os.path.join(DLPFC_DIR, "dlpfc_subtype_heatmap.png"),
    figsize=(14, 8),
)
plt.show()

fig_gene = generate_gene_heatmap(
    best_dlpfc["characterization"],
    output_path=os.path.join(DLPFC_DIR, "dlpfc_gene_heatmap.png"),
    figsize=(16, 10),
    top_n=15,
)
plt.show()

In [ ]:
# ── 11e. Benchmark on DLPFC subset ───────────────────────────────────────────

print("Running benchmark comparison (DLPFC SCZ+CTL only)...")
dlpfc_bench = run_benchmark_comparison(
    gene_burdens=dlpfc_scz_ctl_expr,
    pathway_scores=dlpfc_scz_ctl_scores,
    pathways=scz_pathways,
    n_clusters=best_dlpfc_k,
    seed=SEED,
)
print("\n" + dlpfc_bench.format_report())

# Visualize
dlpfc_methods = list(dlpfc_bench.method_results.keys())
dlpfc_sils = [dlpfc_bench.method_results[m].silhouette for m in dlpfc_methods]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71" if m == dlpfc_bench.best_method else "#3498db" for m in dlpfc_methods]
bars = ax.barh(dlpfc_methods, dlpfc_sils, color=colors)
ax.set_xlabel("Silhouette Score (higher is better)")
ax.set_title(f"DLPFC Benchmark (k={best_dlpfc_k})")
for bar, val in zip(bars, dlpfc_sils):
    ax.text(max(bar.get_width() + 0.005, 0.01), bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(DLPFC_DIR, "dlpfc_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── 11f. Cross-region comparison ─────────────────────────────────────────────

from sklearn.metrics import adjusted_rand_score

# Run quick subtyping per region (all 4 diagnoses) to compare structure
region_results = {}
for region in ["ACC", "DLPFC", "NAc"]:
    region_mask = metadata.loc[pathway_scores_scz.index, "brain_region"] == region
    r_scores = pathway_scores_scz.loc[region_mask]
    r_expr = gene_expression.loc[region_mask]
    r_meta = metadata.loc[r_scores.index]
    
    # Subset to SCZ+CTL
    scz_ctl_r_mask = r_meta["diagnosis"].isin(["SCZ", "Control"])
    r_scores_sc = r_scores.loc[scz_ctl_r_mask]
    r_meta_sc = r_meta.loc[scz_ctl_r_mask]
    
    if len(r_scores_sc) < 10:
        print(f"Skipping {region} — too few SCZ+CTL samples ({len(r_scores_sc)})")
        continue
    
    # Cap k-range
    r_max_k = min(4, len(r_scores_sc) // 8)
    r_max_k = max(r_max_k, 2)
    
    # BIC k-selection
    r_selection = select_n_clusters(
        data=r_scores_sc.values,
        k_range=list(range(2, r_max_k + 1)),
        method="bic",
        seed=SEED,
    )
    
    # Cluster at optimal k
    r_clustering = run_clustering(
        data=r_scores_sc.values,
        n_clusters=r_selection.optimal_k,
        algorithm=ClusteringAlgorithm.GMM,
        seed=SEED,
    )
    
    region_results[region] = {
        "n_samples": len(r_scores_sc),
        "n_scz": int((r_meta_sc["diagnosis"] == "SCZ").sum()),
        "n_ctl": int((r_meta_sc["diagnosis"] == "Control").sum()),
        "optimal_k": r_selection.optimal_k,
        "silhouette": float(r_clustering.silhouette),
        "labels": r_clustering.labels,
        "scores": r_scores_sc,
    }
    
    ct = pd.crosstab(
        pd.Series(r_clustering.labels, index=r_meta_sc.index, name="subtype"),
        r_meta_sc["diagnosis"],
    )
    print(f"\n{region} (n={len(r_scores_sc)}, k={r_selection.optimal_k}, "
          f"silhouette={r_clustering.silhouette:.3f}):")
    print(ct)

# Cross-region ARI (for regions with same optimal k or forced k=2)
print(f"\n--- Cross-Region Agreement ---")
regions_done = sorted(region_results.keys())
for i, r1 in enumerate(regions_done):
    for r2 in regions_done[i+1:]:
        # Use common samples if any (unlikely across regions)
        # Instead, compare subtype proportions
        r1_info = region_results[r1]
        r2_info = region_results[r2]
        print(f"  {r1} (k={r1_info['optimal_k']}, sil={r1_info['silhouette']:.3f}) vs "
              f"{r2} (k={r2_info['optimal_k']}, sil={r2_info['silhouette']:.3f})")

In [ ]:
# Visualize pathway score profiles across regions
fig, axes = plt.subplots(1, len(region_results), figsize=(7 * len(region_results), 6))
if len(region_results) == 1:
    axes = [axes]

for i, (region, r_info) in enumerate(sorted(region_results.items())):
    r_mask = metadata.loc[pathway_scores_scz.index, "brain_region"] == region
    r_data = pathway_scores_scz.loc[r_mask]
    r_meta_all = metadata.loc[r_data.index]
    
    scz_mask = r_meta_all["diagnosis"] == "SCZ"
    ctl_mask = r_meta_all["diagnosis"] == "Control"
    
    means_scz = r_data.loc[scz_mask].mean() if scz_mask.any() else pd.Series(0, index=r_data.columns)
    means_ctl = r_data.loc[ctl_mask].mean() if ctl_mask.any() else pd.Series(0, index=r_data.columns)
    diff = means_scz - means_ctl
    
    colors = ["coral" if v > 0 else "steelblue" for v in diff.values]
    axes[i].barh(range(len(diff)), diff.values, color=colors)
    axes[i].set_yticks(range(len(diff)))
    axes[i].set_yticklabels([p.replace("_", " ") for p in diff.index], fontsize=8)
    axes[i].set_xlabel("SCZ - Control (score diff)")
    axes[i].set_title(f"{region} (n={r_info['n_samples']})")
    axes[i].axvline(x=0, color="black", linewidth=0.5)

plt.suptitle("Pathway Score Differences (SCZ vs Control) by Brain Region",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "region_pathway_diff.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Part 3 Summary + Export ───────────────────────────────────────────────────

# Save DLPFC results
best_dlpfc_meta = dlpfc_all_results[best_dlpfc_k]["meta"]
best_dlpfc_meta.to_csv(os.path.join(DLPFC_DIR, "dlpfc_sample_metadata_with_subtypes.csv"))
dlpfc_scz_ctl_scores.to_csv(os.path.join(DLPFC_DIR, "dlpfc_pathway_scores.csv"))

# Export DLPFC characterization
dlpfc_export = export_characterization(
    dlpfc_all_results[best_dlpfc_k]["characterization"],
    output_dir=DLPFC_DIR,
    formats=["csv"],
)
print("DLPFC characterization files:")
for f in dlpfc_export:
    print(f"  {f}")

# Save DLPFC summary JSON
dlpfc_summary = {
    "analysis": "dlpfc_region_focused",
    "dataset": "GSE80655",
    "brain_region": "DLPFC",
    "n_samples": int(n_dlpfc),
    "n_scz": int(n_dlpfc_scz),
    "n_control": int(n_dlpfc_ctl),
    "k_tested": dlpfc_k_values,
    "best_k": int(best_dlpfc_k),
    "results_by_k": {
        str(k): {
            "silhouette": float(dlpfc_all_results[k]["silhouette"]),
            "n_gates_passed": int(dlpfc_all_results[k]["n_gates_passed"]),
            "all_gates_passed": bool(dlpfc_all_results[k]["validation"].all_passed),
            "validation_gates": [
                {"name": str(g.name), "passed": bool(g.passed),
                 "metric": str(g.metric_name), "value": float(g.metric_value),
                 "threshold": float(g.threshold)}
                for g in dlpfc_all_results[k]["validation"].results
            ],
        }
        for k in dlpfc_k_values
    },
    "benchmark_winner": str(dlpfc_bench.best_method),
    "comparison_to_full_dataset": {
        "full_k": int(optimal_k),
        "full_silhouette": float(clustering.silhouette),
        "full_gates_passed": int(full_gates_passed),
        "dlpfc_k": int(best_dlpfc_k),
        "dlpfc_silhouette": float(dlpfc_all_results[best_dlpfc_k]["silhouette"]),
        "dlpfc_gates_passed": int(dlpfc_all_results[best_dlpfc_k]["n_gates_passed"]),
    },
    "region_comparison": {
        region: {
            "n_samples": int(r["n_samples"]),
            "n_scz": int(r["n_scz"]),
            "n_ctl": int(r["n_ctl"]),
            "optimal_k": int(r["optimal_k"]),
            "silhouette": float(r["silhouette"]),
        }
        for region, r in region_results.items()
    },
    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(DLPFC_DIR, "dlpfc_results_summary.json"), "w") as f:
    json.dump(dlpfc_summary, f, indent=2)

# Update main results_summary.json with benchmark + DLPFC info
with open(os.path.join(OUTPUT_DIR, "results_summary.json")) as f:
    results_summary = json.load(f)

results_summary["benchmark_best_method"] = str(bench_result.best_method)
results_summary["benchmark_ranking"] = [str(r) for r in bench_result.ranking]
results_summary["algorithm_comparison"] = {
    "most_stable": str(algo_comparison.most_stable_algorithm),
    "pairwise_ari": {str(k): float(v) for k, v in algo_comparison.pairwise_ari.items()},
}
results_summary["dlpfc_analysis"] = {
    "best_k": int(best_dlpfc_k),
    "silhouette": float(dlpfc_all_results[best_dlpfc_k]["silhouette"]),
    "gates_passed": int(dlpfc_all_results[best_dlpfc_k]["n_gates_passed"]),
    "benchmark_winner": str(dlpfc_bench.best_method),
}

with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)

# Print summary
val_status = "ALL PASSED" if val_result.all_passed else "SOME FAILED"
dlpfc_val_status = "ALL PASSED" if dlpfc_all_results[best_dlpfc_k]["validation"].all_passed else "SOME FAILED"

print("\n" + "=" * 60)
print("PART 3 COMPLETE: Benchmark, Algorithms & DLPFC Analysis")
print("=" * 60)
print(f"\n--- Full SCZ+CTL Analysis ---")
print(f"  Benchmark winner: {bench_result.best_method}")
print(f"  Algorithm comparison: most stable = {algo_comparison.most_stable_algorithm}")
print(f"  Validation: {val_status}")
print(f"\n--- DLPFC Region-Focused ---")
print(f"  Samples: {n_dlpfc} ({n_dlpfc_scz} SCZ, {n_dlpfc_ctl} Control)")
print(f"  Best k: {best_dlpfc_k}")
print(f"  Silhouette: {dlpfc_all_results[best_dlpfc_k]['silhouette']:.4f}")
print(f"  Validation: {dlpfc_val_status}")
print(f"  Benchmark winner: {dlpfc_bench.best_method}")
print(f"\n--- Region Comparison ---")
for region, r in sorted(region_results.items()):
    print(f"  {region}: n={r['n_samples']}, k={r['optimal_k']}, sil={r['silhouette']:.3f}")
print(f"\nOutputs saved to:")
print(f"  {OUTPUT_DIR}/benchmark_comparison.png")
print(f"  {OUTPUT_DIR}/region_pathway_diff.png")
print(f"  {OUTPUT_DIR}/results_summary.json (updated)")
print(f"  {DLPFC_DIR}/dlpfc_*.png, dlpfc_*.csv, dlpfc_results_summary.json")

## 12. Cross-Disease Pathway Analysis

**Key question:** Do autism-derived pathways stratify schizophrenia patients?

We already scored all samples with both SCZ and ASD pathway sets (Section 6).
Now we:
1. Cluster SCZ+CTL samples on **ASD** pathway scores
2. Compare ASD-pathway subtypes to SCZ-pathway subtypes (ARI)
3. Correlate shared pathways across the two scoring systems
4. Visualize the two pathway spaces side-by-side

A high ARI would suggest shared molecular architecture across disorders.

In [ ]:
# ── 12a. Cluster SCZ+CTL on ASD pathway scores ──────────────────────────────

from sklearn.metrics import adjusted_rand_score

# Subset ASD pathway scores to SCZ+CTL samples
asd_scz_ctl_scores = pathway_scores_asd.loc[scz_ctl_mask]
print(f"ASD pathway scores (SCZ+CTL): {asd_scz_ctl_scores.shape}")
print(f"SCZ pathway scores (SCZ+CTL): {scz_ctl_scores.shape}")

# Drop any ASD pathways with zero variance in the SCZ+CTL subset
asd_var = asd_scz_ctl_scores.var()
low_var = asd_var[asd_var < 1e-10]
if len(low_var) > 0:
    print(f"Dropping {len(low_var)} zero-variance ASD pathways: {list(low_var.index)}")
    asd_scz_ctl_scores = asd_scz_ctl_scores.drop(columns=low_var.index)

# BIC k-selection on ASD pathway scores
asd_max_k = min(7, len(asd_scz_ctl_scores) // 8)
asd_max_k = max(asd_max_k, 2)
asd_k_range = list(range(2, asd_max_k + 1))

asd_selection = select_n_clusters(
    data=asd_scz_ctl_scores.values,
    k_range=asd_k_range,
    method="bic",
    seed=SEED,
)
asd_optimal_k = asd_selection.optimal_k
print(f"\nASD-pathway optimal k (BIC): {asd_optimal_k}")
print(f"  BIC values: {asd_selection.bic_values}")
print(f"  Silhouette values: {asd_selection.silhouette_values}")

# Cluster at optimal k
asd_clustering = run_clustering(
    data=asd_scz_ctl_scores.values,
    n_clusters=asd_optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)
asd_labels = asd_clustering.labels

print(f"\n--- ASD-Pathway Clustering (SCZ+CTL) ---")
print(f"k = {asd_optimal_k}")
print(f"Silhouette: {asd_clustering.silhouette:.4f}")
print(f"Calinski-Harabasz: {asd_clustering.calinski_harabasz:.2f}")
print(f"Davies-Bouldin: {asd_clustering.davies_bouldin:.4f}")

# Subtype sizes
for i in range(asd_optimal_k):
    count = int((asd_labels == i).sum())
    print(f"  ASD-Subtype {i}: {count} ({count / len(asd_labels) * 100:.1f}%)")

# Cross-tab with diagnosis
asd_sub_meta = scz_ctl_meta.copy()
asd_sub_meta["asd_subtype"] = asd_labels
ct_asd = pd.crosstab(asd_sub_meta["asd_subtype"], asd_sub_meta["diagnosis"], margins=True)
print(f"\n--- ASD-Pathway Subtype x Diagnosis ---")
print(ct_asd)

# ── Compare SCZ-pathway vs ASD-pathway subtypes ─────────────────────────────

# ARI: how much do the two pathway-based subtype assignments agree?
cross_ari = adjusted_rand_score(labels, asd_labels)
print(f"\n{'=' * 60}")
print(f"CROSS-DISEASE PATHWAY COMPARISON")
print(f"{'=' * 60}")
print(f"SCZ-pathway subtypes: k={optimal_k}")
print(f"ASD-pathway subtypes: k={asd_optimal_k}")
print(f"\nAdjusted Rand Index (SCZ vs ASD pathways): {cross_ari:.4f}")

if cross_ari > 0.3:
    interp = "STRONG agreement — shared molecular architecture across disorders"
elif cross_ari > 0.1:
    interp = "MODERATE agreement — partial overlap in pathway-based stratification"
else:
    interp = "WEAK agreement — disorder-specific pathway patterns dominate"
print(f"Interpretation: {interp}")

# Cross-tabulate the two subtype systems
ct_cross = pd.crosstab(
    pd.Series(labels, index=scz_ctl_scores.index, name="SCZ_subtype"),
    pd.Series(asd_labels, index=asd_scz_ctl_scores.index, name="ASD_subtype"),
    margins=True,
)
print(f"\n--- SCZ-Pathway Subtypes x ASD-Pathway Subtypes ---")
print(ct_cross)

In [ ]:
# ── 12b. Shared pathway correlation ──────────────────────────────────────────

# For the 8 shared pathways, correlate SCZ scores vs ASD scores
print(f"Shared pathways between SCZ and ASD: {len(shared_pathway_names)}")
print(f"  {shared_pathway_names}")

# Build correlation matrix: for each shared pathway, correlate SCZ score vs ASD score
shared_corr = {}
for pw in shared_pathway_names:
    scz_col = pw if pw in scz_ctl_scores.columns else None
    asd_col = pw if pw in asd_scz_ctl_scores.columns else None
    if scz_col and asd_col:
        corr = np.corrcoef(
            scz_ctl_scores[scz_col].values,
            asd_scz_ctl_scores[asd_col].values,
        )[0, 1]
        shared_corr[pw] = corr
        print(f"  {pw}: r = {corr:.4f}")

# Visualize shared pathway correlations
if shared_corr:
    fig, ax = plt.subplots(figsize=(10, 5))
    pathways_sorted = sorted(shared_corr.keys(), key=lambda x: shared_corr[x], reverse=True)
    corr_vals = [shared_corr[p] for p in pathways_sorted]
    colors = ["#2ecc71" if v > 0.5 else "#f39c12" if v > 0.3 else "#e74c3c" for v in corr_vals]
    
    bars = ax.barh(
        [p.replace("_", " ") for p in pathways_sorted],
        corr_vals, color=colors,
    )
    for bar, val in zip(bars, corr_vals):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"r={val:.3f}", va="center", fontsize=9)
    
    ax.set_xlabel("Pearson Correlation (SCZ vs ASD pathway scores)")
    ax.set_title("Shared Pathway Score Correlation\n(Same pathway scored from SCZ vs ASD gene sets)")
    ax.set_xlim(0, 1.15)
    ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="r=0.5")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CROSS_DISEASE_DIR, "shared_pathway_correlation.png"),
                dpi=150, bbox_inches="tight")
    plt.show()
    
    mean_corr = np.mean(corr_vals)
    print(f"\nMean shared pathway correlation: r = {mean_corr:.4f}")
    if mean_corr > 0.7:
        print("  -> High correlation: SCZ and ASD gene sets capture similar signals")
    elif mean_corr > 0.4:
        print("  -> Moderate correlation: partial overlap in pathway biology")
    else:
        print("  -> Low correlation: disorder-specific gene content drives different signals")

In [ ]:
# ── 12c. Cross-disease PCA visualization ─────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Top-left: SCZ pathway space — colored by SCZ subtypes
scz_emb, scz_pca = compute_dim_reduction(
    pathway_scores=scz_ctl_scores, method=DimReductionMethod.PCA,
    n_components=2, seed=SEED,
)
scatter_colors_scz = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    mask = labels == i
    axes[0, 0].scatter(scz_emb[mask, 0], scz_emb[mask, 1], c=[scatter_colors_scz[i]],
                       label=f"SCZ-Sub {i} (n={int(mask.sum())})", s=60, alpha=0.7,
                       edgecolors="k", linewidth=0.5)
axes[0, 0].set_xlabel(f"PC1 ({scz_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[0, 0].set_ylabel(f"PC2 ({scz_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[0, 0].set_title("SCZ Pathway Space — SCZ Subtypes")
axes[0, 0].legend(fontsize=8)

# Top-right: ASD pathway space — colored by ASD subtypes
asd_emb, asd_pca = compute_dim_reduction(
    pathway_scores=asd_scz_ctl_scores, method=DimReductionMethod.PCA,
    n_components=2, seed=SEED,
)
scatter_colors_asd = plt.cm.Set1(np.linspace(0, 1, asd_optimal_k))
for i in range(asd_optimal_k):
    mask = asd_labels == i
    axes[0, 1].scatter(asd_emb[mask, 0], asd_emb[mask, 1], c=[scatter_colors_asd[i]],
                       label=f"ASD-Sub {i} (n={int(mask.sum())})", s=60, alpha=0.7,
                       edgecolors="k", linewidth=0.5)
axes[0, 1].set_xlabel(f"PC1 ({asd_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[0, 1].set_ylabel(f"PC2 ({asd_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[0, 1].set_title("ASD Pathway Space — ASD Subtypes")
axes[0, 1].legend(fontsize=8)

# Bottom-left: SCZ pathway space — colored by diagnosis
dx_colors_map = {"SCZ": "tomato", "Control": "steelblue"}
for dx, color in dx_colors_map.items():
    mask = scz_ctl_meta["diagnosis"].values == dx
    axes[1, 0].scatter(scz_emb[mask, 0], scz_emb[mask, 1], c=color,
                       label=dx, s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[1, 0].set_xlabel(f"PC1 ({scz_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[1, 0].set_ylabel(f"PC2 ({scz_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[1, 0].set_title("SCZ Pathway Space — Diagnosis")
axes[1, 0].legend(fontsize=8)

# Bottom-right: ASD pathway space — colored by diagnosis
for dx, color in dx_colors_map.items():
    mask = scz_ctl_meta["diagnosis"].values == dx
    axes[1, 1].scatter(asd_emb[mask, 0], asd_emb[mask, 1], c=color,
                       label=dx, s=60, alpha=0.7, edgecolors="k", linewidth=0.5)
axes[1, 1].set_xlabel(f"PC1 ({asd_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[1, 1].set_ylabel(f"PC2 ({asd_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[1, 1].set_title("ASD Pathway Space — Diagnosis")
axes[1, 1].legend(fontsize=8)

plt.suptitle(f"Cross-Disease Pathway Comparison (SCZ+CTL, n={len(scz_ctl_scores)})\n"
             f"SCZ-pathway subtypes (k={optimal_k}) vs ASD-pathway subtypes (k={asd_optimal_k}) — "
             f"ARI = {cross_ari:.3f}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CROSS_DISEASE_DIR, "cross_disease_pca.png"), dpi=150, bbox_inches="tight")
plt.show()

## 13. Multi-Diagnosis Pooled Clustering

**Key question:** Do molecular subtypes cut across diagnostic boundaries?

Pool **all 4 diagnoses** (SCZ, BD, MDD, Control) and cluster on SCZ pathway scores.
If subtypes are trans-diagnostic, we expect subtypes to contain a mix of diagnoses;
if they are diagnosis-specific, each subtype should be enriched for one diagnosis.

We test this formally with a **chi-squared test** for subtype-diagnosis independence.

In [ ]:
# ── 13a. Pool all diagnoses + BIC k-selection ────────────────────────────────

# Use SCZ pathway scores for all 281 samples
all_scores = pathway_scores_scz.copy()
all_meta = metadata.loc[all_scores.index].copy()
n_all = len(all_scores)

print("=" * 60)
print("MULTI-DIAGNOSIS POOLED CLUSTERING")
print("=" * 60)
print(f"\nTotal samples: {n_all}")
print(f"Diagnosis breakdown:")
for dx in ["SCZ", "BD", "MDD", "Control"]:
    n = (all_meta["diagnosis"] == dx).sum()
    print(f"  {dx}: {n} ({n / n_all * 100:.1f}%)")

# Cap k-range
pool_max_k = min(8, n_all // 10)
pool_max_k = max(pool_max_k, 2)
pool_k_range = list(range(2, pool_max_k + 1))
print(f"\nk-range for BIC: {pool_k_range}")

# BIC k-selection
pool_selection = select_n_clusters(
    data=all_scores.values,
    k_range=pool_k_range,
    method="bic",
    seed=SEED,
)
pool_optimal_k = pool_selection.optimal_k
print(f"Optimal k (BIC): {pool_optimal_k}")

# Plot BIC + silhouette
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ks = sorted(pool_selection.bic_values.keys())

ax1.plot(ks, [pool_selection.bic_values[k] for k in ks], "bo-", linewidth=2)
ax1.axvline(x=pool_optimal_k, color="red", linestyle="--", label=f"Optimal k={pool_optimal_k}")
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("BIC (lower is better)")
ax1.set_title("All Diagnoses: BIC")
ax1.legend()

ax2.plot(ks, [pool_selection.silhouette_values[k] for k in ks], "go-", linewidth=2)
ax2.axvline(x=pool_optimal_k, color="red", linestyle="--", label=f"Optimal k={pool_optimal_k}")
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("All Diagnoses: Silhouette")
ax2.legend()

plt.suptitle(f"GSE80655 Multi-Diagnosis: Cluster Selection (n={n_all})",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CROSS_DISEASE_DIR, "pooled_model_selection.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# Cluster at optimal k
pool_clustering = run_clustering(
    data=all_scores.values,
    n_clusters=pool_optimal_k,
    algorithm=ClusteringAlgorithm.GMM,
    seed=SEED,
)
pool_labels = pool_clustering.labels

print(f"\n--- Pooled Clustering Results ---")
print(f"k = {pool_optimal_k}")
print(f"Silhouette: {pool_clustering.silhouette:.4f}")
print(f"Calinski-Harabasz: {pool_clustering.calinski_harabasz:.2f}")
print(f"Davies-Bouldin: {pool_clustering.davies_bouldin:.4f}")

all_meta["pool_subtype"] = pool_labels
print(f"\nSubtype sizes:")
for i in range(pool_optimal_k):
    count = int((pool_labels == i).sum())
    print(f"  Subtype {i}: {count} ({count / n_all * 100:.1f}%)")

# Cross-tab: subtype x diagnosis
ct_pool_dx = pd.crosstab(all_meta["pool_subtype"], all_meta["diagnosis"], margins=True)
print(f"\n--- Pooled Subtype x Diagnosis ---")
print(ct_pool_dx)

# Cross-tab: subtype x brain region
ct_pool_region = pd.crosstab(all_meta["pool_subtype"], all_meta["brain_region"], margins=True)
print(f"\n--- Pooled Subtype x Brain Region ---")
print(ct_pool_region)

In [ ]:
# ── 13b. Chi-squared test for subtype-diagnosis independence ─────────────────

from scipy.stats import chi2_contingency

# Test: are subtypes independent of diagnosis?
ct_for_chi2 = pd.crosstab(all_meta["pool_subtype"], all_meta["diagnosis"])
chi2, p_value, dof, expected = chi2_contingency(ct_for_chi2)

print("=" * 60)
print("CHI-SQUARED TEST: Subtype ~ Diagnosis Independence")
print("=" * 60)
print(f"\nObserved frequencies:")
print(ct_for_chi2)
print(f"\nExpected frequencies (under independence):")
print(pd.DataFrame(expected, index=ct_for_chi2.index, columns=ct_for_chi2.columns).round(1))
print(f"\nChi-squared statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("\nResult: SIGNIFICANT (p < 0.05)")
    print("  -> Subtypes are NOT independent of diagnosis")
    print("  -> Some subtypes are enriched for specific diagnoses")
else:
    print("\nResult: NOT significant (p >= 0.05)")
    print("  -> Subtypes appear independent of diagnosis")
    print("  -> Trans-diagnostic molecular subtypes")

# Per-subtype diagnosis enrichment (observed / expected ratio)
print(f"\n--- Per-Subtype Diagnosis Enrichment (Observed/Expected) ---")
enrichment = ct_for_chi2.values / expected
enrichment_df = pd.DataFrame(enrichment, index=ct_for_chi2.index, columns=ct_for_chi2.columns)
print(enrichment_df.round(2))

print(f"\n--- Interpretation ---")
for sub_idx in enrichment_df.index:
    max_dx = enrichment_df.loc[sub_idx].idxmax()
    max_ratio = enrichment_df.loc[sub_idx].max()
    if max_ratio > 1.5:
        print(f"  Subtype {sub_idx}: enriched for {max_dx} ({max_ratio:.2f}x expected)")
    elif max_ratio > 1.2:
        print(f"  Subtype {sub_idx}: mildly enriched for {max_dx} ({max_ratio:.2f}x)")
    else:
        print(f"  Subtype {sub_idx}: no strong diagnosis enrichment (max {max_ratio:.2f}x for {max_dx})")

# Chi-squared for subtype ~ brain region
ct_region_chi2 = pd.crosstab(all_meta["pool_subtype"], all_meta["brain_region"])
chi2_region, p_region, dof_region, _ = chi2_contingency(ct_region_chi2)
print(f"\n--- Subtype ~ Brain Region ---")
print(f"Chi-squared: {chi2_region:.4f}, p = {p_region:.4e}, dof = {dof_region}")
if p_region < 0.05:
    print("  -> Subtypes are region-dependent (p < 0.05)")
else:
    print("  -> Subtypes are region-independent (p >= 0.05)")

In [ ]:
# ── 13c. Multi-diagnosis visualization ───────────────────────────────────────

# PCA embedding of all samples
pool_emb, pool_pca = compute_dim_reduction(
    pathway_scores=all_scores, method=DimReductionMethod.PCA,
    n_components=2, seed=SEED,
)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Plot 1: Colored by pooled subtype
sub_colors = plt.cm.Set2(np.linspace(0, 1, pool_optimal_k))
for i in range(pool_optimal_k):
    mask = pool_labels == i
    axes[0].scatter(pool_emb[mask, 0], pool_emb[mask, 1], c=[sub_colors[i]],
                    label=f"Subtype {i} (n={int(mask.sum())})", s=50, alpha=0.7,
                    edgecolors="k", linewidth=0.3)
axes[0].set_xlabel(f"PC1 ({pool_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pool_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[0].set_title("Colored by Molecular Subtype")
axes[0].legend(fontsize=7)

# Plot 2: Colored by diagnosis
all_dx_colors = {"SCZ": "tomato", "BD": "orange", "MDD": "mediumpurple", "Control": "steelblue"}
for dx, color in all_dx_colors.items():
    mask = all_meta["diagnosis"].values == dx
    if mask.any():
        axes[1].scatter(pool_emb[mask, 0], pool_emb[mask, 1], c=color,
                        label=f"{dx} (n={int(mask.sum())})", s=50, alpha=0.7,
                        edgecolors="k", linewidth=0.3)
axes[1].set_xlabel(f"PC1 ({pool_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pool_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[1].set_title("Colored by Diagnosis")
axes[1].legend(fontsize=7)

# Plot 3: Colored by brain region
region_colors = {"ACC": "#2ecc71", "DLPFC": "#e74c3c", "NAc": "#3498db"}
for region, color in region_colors.items():
    mask = all_meta["brain_region"].values == region
    if mask.any():
        axes[2].scatter(pool_emb[mask, 0], pool_emb[mask, 1], c=color,
                        label=f"{region} (n={int(mask.sum())})", s=50, alpha=0.7,
                        edgecolors="k", linewidth=0.3)
axes[2].set_xlabel(f"PC1 ({pool_pca['explained_variance_ratio'][0]*100:.1f}%)")
axes[2].set_ylabel(f"PC2 ({pool_pca['explained_variance_ratio'][1]*100:.1f}%)")
axes[2].set_title("Colored by Brain Region")
axes[2].legend(fontsize=7)

plt.suptitle(f"GSE80655 Multi-Diagnosis Pooled Clustering (k={pool_optimal_k}, n={n_all})\n"
             f"Chi-squared p={p_value:.2e} (subtype ~ diagnosis)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CROSS_DISEASE_DIR, "pooled_pca_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

# Diagnosis composition per subtype (stacked bar)
fig, ax = plt.subplots(figsize=(10, 6))
ct_norm = ct_for_chi2.div(ct_for_chi2.sum(axis=1), axis=0)
ct_norm.plot(kind="bar", stacked=True, ax=ax,
             color=[all_dx_colors.get(c, "gray") for c in ct_for_chi2.columns])
ax.set_xlabel("Molecular Subtype")
ax.set_ylabel("Proportion")
ax.set_title(f"Diagnosis Composition per Subtype (k={pool_optimal_k})")
ax.legend(title="Diagnosis", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xticklabels([f"Subtype {i}" for i in ct_for_chi2.index], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(CROSS_DISEASE_DIR, "pooled_diagnosis_composition.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Part 4 Summary + Cross-Disease Export ─────────────────────────────────────

# Save pooled clustering metadata
all_meta.to_csv(os.path.join(CROSS_DISEASE_DIR, "pooled_metadata_with_subtypes.csv"))

# Save ASD-pathway subtype assignments for SCZ+CTL
asd_sub_meta.to_csv(os.path.join(CROSS_DISEASE_DIR, "asd_pathway_subtypes_scz_ctl.csv"))

# Save cross-disease summary JSON
cross_disease_summary = {
    "analysis": "cross_disease",
    "dataset": "GSE80655",
    "citation": "Ramaker et al. 2017, Genome Medicine",

    "cross_pathway_comparison": {
        "scz_pathway_k": int(optimal_k),
        "scz_pathway_silhouette": float(clustering.silhouette),
        "asd_pathway_k": int(asd_optimal_k),
        "asd_pathway_silhouette": float(asd_clustering.silhouette),
        "cross_pathway_ari": float(cross_ari),
        "shared_pathways": shared_pathway_names,
        "shared_pathway_correlations": {str(k): float(v) for k, v in shared_corr.items()},
        "mean_shared_correlation": float(np.mean(list(shared_corr.values()))) if shared_corr else 0.0,
    },

    "multi_diagnosis_pooled": {
        "n_samples": int(n_all),
        "diagnosis_counts": {
            dx: int((all_meta["diagnosis"] == dx).sum())
            for dx in ["SCZ", "BD", "MDD", "Control"]
        },
        "optimal_k": int(pool_optimal_k),
        "silhouette": float(pool_clustering.silhouette),
        "calinski_harabasz": float(pool_clustering.calinski_harabasz),
        "davies_bouldin": float(pool_clustering.davies_bouldin),
        "chi_squared_diagnosis": {
            "statistic": float(chi2),
            "p_value": float(p_value),
            "dof": int(dof),
            "significant": bool(p_value < 0.05),
        },
        "chi_squared_region": {
            "statistic": float(chi2_region),
            "p_value": float(p_region),
            "dof": int(dof_region),
            "significant": bool(p_region < 0.05),
        },
        "subtype_sizes": {
            str(i): int((pool_labels == i).sum()) for i in range(pool_optimal_k)
        },
        "diagnosis_enrichment": {
            str(sub_idx): {
                str(dx): float(enrichment_df.loc[sub_idx, dx])
                for dx in enrichment_df.columns
            }
            for sub_idx in enrichment_df.index
        },
    },

    "framework_version": "0.3.0",
    "seed": SEED,
}

with open(os.path.join(CROSS_DISEASE_DIR, "cross_disease_summary.json"), "w") as f:
    json.dump(cross_disease_summary, f, indent=2)

# Update main results_summary.json
with open(os.path.join(OUTPUT_DIR, "results_summary.json")) as f:
    results_summary = json.load(f)

results_summary["cross_disease"] = {
    "asd_pathway_k": int(asd_optimal_k),
    "asd_pathway_silhouette": float(asd_clustering.silhouette),
    "cross_pathway_ari": float(cross_ari),
    "mean_shared_correlation": float(np.mean(list(shared_corr.values()))) if shared_corr else 0.0,
}
results_summary["multi_diagnosis_pooled"] = {
    "n_samples": int(n_all),
    "optimal_k": int(pool_optimal_k),
    "silhouette": float(pool_clustering.silhouette),
    "chi2_p_value": float(p_value),
    "chi2_significant": bool(p_value < 0.05),
}

with open(os.path.join(OUTPUT_DIR, "results_summary.json"), "w") as f:
    json.dump(results_summary, f, indent=2)

# Print summary
print("\n" + "=" * 60)
print("PART 4 COMPLETE: Cross-Disease & Multi-Diagnosis Analysis")
print("=" * 60)

print(f"\n--- Section 12: Cross-Disease Pathway Analysis ---")
print(f"  SCZ-pathway subtypes: k={optimal_k}, sil={clustering.silhouette:.4f}")
print(f"  ASD-pathway subtypes: k={asd_optimal_k}, sil={asd_clustering.silhouette:.4f}")
print(f"  Cross-pathway ARI: {cross_ari:.4f}")
mean_shared = np.mean(list(shared_corr.values())) if shared_corr else 0.0
print(f"  Shared pathway mean correlation: r={mean_shared:.4f}")

print(f"\n--- Section 13: Multi-Diagnosis Pooled Clustering ---")
print(f"  All diagnoses: {n_all} samples")
print(f"  Optimal k: {pool_optimal_k}")
print(f"  Silhouette: {pool_clustering.silhouette:.4f}")
print(f"  Chi-squared (subtype ~ diagnosis): p={p_value:.4e}")
chi2_interp = "DEPENDENT" if p_value < 0.05 else "INDEPENDENT"
print(f"  Subtype-diagnosis: {chi2_interp}")
chi2_region_interp = "DEPENDENT" if p_region < 0.05 else "INDEPENDENT"
print(f"  Subtype-region: {chi2_region_interp}")

print(f"\nOutputs saved to: {CROSS_DISEASE_DIR}/")
print(f"  - cross_disease_summary.json")
print(f"  - pooled_metadata_with_subtypes.csv")
print(f"  - asd_pathway_subtypes_scz_ctl.csv")
print(f"  - shared_pathway_correlation.png")
print(f"  - cross_disease_pca.png")
print(f"  - pooled_model_selection.png")
print(f"  - pooled_pca_scatter.png")
print(f"  - pooled_diagnosis_composition.png")
print(f"  {OUTPUT_DIR}/results_summary.json (updated)")

## 14. Cross-Cohort Projection: GSE28521 → GSE80655

We project autism subtypes discovered in GSE28521 (Voineagu et al. 2011, frontal
cortex microarray) onto the GSE80655 DLPFC SCZ+CTL samples using ASD pathway scores.

**Approach:**
1. Re-download and process GSE28521 (self-contained)
2. Score GSE28521 frontal cortex with ASD pathways
3. Train a GMM on the GSE28521 ASD pathway scores
4. Apply the same scaler + GMM to GSE80655 DLPFC SCZ+CTL ASD scores
5. Compare projected subtype profiles between cohorts

This tests whether ASD-derived molecular subtypes generalize across
disorders and platforms (microarray → RNA-seq).

In [ ]:
# ── 14a. Rebuild GSE28521 discovery model (ASD pathways) ─────────────────────

# Try loading from notebook 10 outputs first; fall back to re-download
gse28521_fc_path = "./outputs/gse28521/frontal_cortex/fc_pathway_scores.csv"
gse28521_meta_path = "./outputs/gse28521/frontal_cortex/fc_sample_metadata_with_subtypes.csv"

if os.path.exists(gse28521_fc_path) and os.path.exists(gse28521_meta_path):
    print("Loading GSE28521 frontal cortex results from notebook 10 outputs...")
    disc_fc_expr_raw = pd.read_csv(
        gse28521_fc_path.replace("fc_pathway_scores", "../gene_expression_processed"),
        index_col=0,
    ) if os.path.exists(gse28521_fc_path.replace("fc_pathway_scores", "../gene_expression_processed")) else None
    # We need raw expression to re-score with ASD pathways, so always re-download
    disc_fc_expr_raw = None  # force re-download

# Always re-download to guarantee self-contained notebook
print("Downloading and processing GSE28521 for cross-cohort projection...")
gse_disc = GEOparse.get_GEO(geo="GSE28521", destdir=DATA_DIR, silent=True)

# ── Metadata ──────────────────────────────────────────────────────────────
disc_meta_rows = []
for gsm_name, gsm in gse_disc.gsms.items():
    title = gsm.metadata.get("title", [""])[0]
    parts = title.split("_")
    diagnosis = "ASD" if parts[0] == "A" else "Control" if parts[0] == "C" else "Unknown"
    region_map = {"C": "Cerebellum", "F": "Frontal_Cortex", "T": "Temporal_Cortex"}
    region = region_map.get(parts[-1], "Unknown") if len(parts) >= 3 else "Unknown"
    disc_meta_rows.append({"sample_id": gsm_name, "diagnosis": diagnosis, "brain_region": region})
disc_meta = pd.DataFrame(disc_meta_rows).set_index("sample_id")

# ── Expression matrix ──────────────────────────────────────────────────────
disc_expr_df = gse_disc.pivot_samples("VALUE").apply(pd.to_numeric, errors="coerce").dropna(how="all")
disc_gpl = list(gse_disc.gpls.values())[0]

# Find gene symbol column
disc_sym_col = None
for col in ["Symbol", "Gene Symbol", "GENE_SYMBOL", "Gene_Symbol"]:
    if col in disc_gpl.table.columns:
        disc_sym_col = col
        break
if disc_sym_col is None:
    for col in disc_gpl.table.columns:
        if "symbol" in col.lower():
            disc_sym_col = col
            break

disc_p2g = disc_gpl.table.set_index("ID")[disc_sym_col].dropna()
disc_p2g = disc_p2g[disc_p2g.str.strip() != ""]
disc_common = disc_expr_df.index.intersection(disc_p2g.index)
disc_expr_df = disc_expr_df.loc[disc_common]
disc_expr_df.index = disc_p2g.loc[disc_common].values
disc_expr_df = disc_expr_df.groupby(disc_expr_df.index).mean()
disc_gene_expr = disc_expr_df.T
disc_gene_expr = disc_gene_expr.loc[:, disc_gene_expr.var() > 0]

# Fill NaN
if disc_gene_expr.isna().any().any():
    n_nan = disc_gene_expr.isna().sum().sum()
    disc_gene_expr = disc_gene_expr.fillna(disc_gene_expr.median())
    print(f"Filled {n_nan} NaN values in GSE28521 expression data")

# Subset to frontal cortex
fc_disc_mask = disc_meta.loc[disc_gene_expr.index, "brain_region"] == "Frontal_Cortex"
disc_fc_expr = disc_gene_expr.loc[fc_disc_mask]
disc_fc_meta = disc_meta.loc[fc_disc_mask]
print(f"GSE28521 frontal cortex: {disc_fc_expr.shape[0]} samples x {disc_fc_expr.shape[1]} genes")
print(f"  ASD: {(disc_fc_meta['diagnosis'] == 'ASD').sum()}, Control: {(disc_fc_meta['diagnosis'] == 'Control').sum()}")

# ── Score with ASD pathways ────────────────────────────────────────────────
disc_asd_scoring = score_pathways_from_expression(
    gene_expression=disc_fc_expr,
    pathways=asd_pathways,
    method=ExpressionScoringMethod.SSGSEA,
    min_genes_per_pathway=2,
    seed=SEED,
)
disc_asd_scores = disc_asd_scoring.pathway_scores

# Handle NaN in pathway scores
if disc_asd_scores.isna().any().any():
    n_nan = disc_asd_scores.isna().sum().sum()
    print(f"Filling {n_nan} NaN in discovery ASD pathway scores")
    disc_asd_scores = disc_asd_scores.fillna(disc_asd_scores.median())
    disc_asd_scores = disc_asd_scores.dropna(axis=1, how="all")

# Cluster at k=2
disc_asd_clustering = run_clustering(
    data=disc_asd_scores.values, n_clusters=2,
    algorithm=ClusteringAlgorithm.GMM, seed=SEED,
)
disc_asd_labels = disc_asd_clustering.labels

print(f"\nGSE28521 ASD-pathway clustering (k=2):")
print(f"  Silhouette: {disc_asd_clustering.silhouette:.4f}")
for i in range(2):
    n = int((disc_asd_labels == i).sum())
    print(f"  Subtype {i}: {n} samples")
print(f"\nDiscovery ASD scores: {disc_asd_scores.shape}")
print(f"Discovery cohort (GSE28521 frontal cortex) ready for projection.")

In [ ]:
# ── 14b. Train GMM and project GSE80655 DLPFC ────────────────────────────────

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

# ── Prepare GSE80655 target: DLPFC SCZ+CTL ASD pathway scores ─────────────
dlpfc_scz_ctl_mask = (
    (metadata["diagnosis"].isin(["SCZ", "Control"])) &
    (metadata["brain_region"] == "DLPFC")
)
target_asd_scores = pathway_scores_asd.loc[dlpfc_scz_ctl_mask]
target_meta = metadata.loc[dlpfc_scz_ctl_mask].copy()
print(f"GSE80655 DLPFC SCZ+CTL: {len(target_asd_scores)} samples")
print(f"  SCZ: {(target_meta['diagnosis'] == 'SCZ').sum()}, Control: {(target_meta['diagnosis'] == 'Control').sum()}")

# ── Align pathways ─────────────────────────────────────────────────────────
common_asd_pw = sorted(set(disc_asd_scores.columns) & set(target_asd_scores.columns))
print(f"\nCommon ASD pathways: {len(common_asd_pw)} of {len(disc_asd_scores.columns)} discovery / {len(target_asd_scores.columns)} target")

disc_aligned = disc_asd_scores[common_asd_pw]
target_aligned = target_asd_scores[common_asd_pw]

# ── Z-score normalize: fit on discovery, transform both ───────────────────
scaler = StandardScaler()
disc_scaled = scaler.fit_transform(disc_aligned)
target_scaled = scaler.transform(target_aligned)

# ── Train GMM on discovery (GSE28521 FC ASD scores) ───────────────────────
gmm = GaussianMixture(
    n_components=2,
    covariance_type="full",
    n_init=10,
    reg_covar=1e-6,
    random_state=SEED,
)
gmm.fit(disc_scaled)
disc_predicted = gmm.predict(disc_scaled)
disc_probs = gmm.predict_proba(disc_scaled)

# ── Project target samples ────────────────────────────────────────────────
target_predicted = gmm.predict(target_scaled)
target_probs = gmm.predict_proba(target_scaled)

# ── Self-consistency check ────────────────────────────────────────────────
disc_self_ari = adjusted_rand_score(disc_asd_labels, disc_predicted)
print(f"\nDiscovery self-ARI (original vs re-trained GMM): {disc_self_ari:.4f}")
print(f"Discovery predicted: {pd.Series(disc_predicted).value_counts().to_dict()}")
print(f"Target projected:    {pd.Series(target_predicted).value_counts().to_dict()}")

# ── Add projected labels to target metadata ───────────────────────────────
target_meta["projected_subtype"] = target_predicted
target_meta["prob_subtype_0"] = target_probs[:, 0]
n_prob_cols = target_probs.shape[1]
if n_prob_cols > 1:
    target_meta["prob_subtype_1"] = target_probs[:, 1]

# ── Cross-tab ─────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"CROSS-COHORT PROJECTION: GSE28521 (ASD) → GSE80655 (SCZ+CTL)")
print(f"{'=' * 60}")

ct_proj = pd.crosstab(target_meta["projected_subtype"], target_meta["diagnosis"], margins=True)
print(f"\nProjected subtype x diagnosis:")
print(ct_proj)

# Composition per projected subtype
print(f"\n--- Projected subtype composition ---")
for sub in sorted(target_meta["projected_subtype"].unique()):
    mask = target_meta["projected_subtype"] == sub
    n_total = int(mask.sum())
    n_scz_proj = int((target_meta.loc[mask, "diagnosis"] == "SCZ").sum())
    n_ctl_proj = n_total - n_scz_proj
    purity = n_scz_proj / n_total * 100 if n_total > 0 else 0
    print(f"  Projected Subtype {sub}: {n_scz_proj} SCZ + {n_ctl_proj} Control ({purity:.0f}% SCZ)")

In [ ]:
# ── 14c. Pathway profile comparison + projection PCA ─────────────────────────

# Mean pathway profiles per subtype in both cohorts
disc_profiles = disc_aligned.groupby(disc_asd_labels).mean()
target_profiles = target_aligned.groupby(target_predicted).mean()

print(f"Discovery subtypes: {list(disc_profiles.index)}")
print(f"Target projected subtypes: {list(target_profiles.index)}")

# Check if both subtypes are present in the projection
proj_has_both = len(target_profiles) >= 2

if proj_has_both:
    # Both subtypes present — compare profiles directly
    for sub_id in sorted(disc_profiles.index):
        if sub_id in target_profiles.index:
            rho, pval = spearmanr(disc_profiles.loc[sub_id], target_profiles.loc[sub_id])
            print(f"\n  Subtype {sub_id} profile correlation (discovery vs target):")
            print(f"    Spearman rho = {rho:.4f}, p = {pval:.2e}")
    proj_ari_note = "Both subtypes projected"
else:
    # Only one subtype in projection
    present_id = list(target_profiles.index)[0]
    absent_id = 1 - present_id
    print(f"\nWARNING: All target samples projected to subtype {present_id}")
    print(f"  Subtype {absent_id} has no target samples")
    val_profile = target_profiles.loc[present_id]
    for disc_id in sorted(disc_profiles.index):
        rho, pval = spearmanr(disc_profiles.loc[disc_id], val_profile)
        print(f"\n  Target (subtype {present_id}) vs Discovery subtype {disc_id}:")
        print(f"    Spearman rho = {rho:.4f}, p = {pval:.2e}")
    proj_ari_note = f"Single-subtype projection (all → subtype {present_id})"

# ── PCA visualization ─────────────────────────────────────────────────────
# Combine discovery and target in shared PCA space
combined_scaled = np.vstack([disc_scaled, target_scaled])
from sklearn.decomposition import PCA as skPCA
pca_proj = skPCA(n_components=2, random_state=SEED)
emb = pca_proj.fit_transform(combined_scaled)

n_disc = len(disc_scaled)
disc_emb = emb[:n_disc]
target_emb = emb[n_disc:]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Colored by cohort
axes[0].scatter(disc_emb[:, 0], disc_emb[:, 1], c="#3498db", marker="o",
                label=f"GSE28521 (n={n_disc})", s=60, alpha=0.7, edgecolors="k", linewidth=0.3)
axes[0].scatter(target_emb[:, 0], target_emb[:, 1], c="#e74c3c", marker="s",
                label=f"GSE80655 (n={len(target_scaled)})", s=60, alpha=0.7, edgecolors="k", linewidth=0.3)
axes[0].set_xlabel(f"PC1 ({pca_proj.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca_proj.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title("Colored by Cohort")
axes[0].legend(fontsize=9)

# Plot 2: Colored by projected subtype
sub_colors = ["#2ecc71", "#9b59b6", "#f39c12", "#e74c3c"]
for sub_id in sorted(set(disc_asd_labels) | set(target_predicted)):
    color = sub_colors[sub_id % len(sub_colors)]
    # Discovery
    d_mask = disc_asd_labels == sub_id
    if d_mask.any():
        axes[1].scatter(disc_emb[d_mask, 0], disc_emb[d_mask, 1], c=color, marker="o",
                        label=f"Disc sub-{sub_id} (n={int(d_mask.sum())})", s=60, alpha=0.6,
                        edgecolors="k", linewidth=0.3)
    # Target
    t_mask = target_predicted == sub_id
    if t_mask.any():
        axes[1].scatter(target_emb[t_mask, 0], target_emb[t_mask, 1], c=color, marker="s",
                        label=f"Tgt sub-{sub_id} (n={int(t_mask.sum())})", s=60, alpha=0.6,
                        edgecolors="k", linewidth=0.3)

axes[1].set_xlabel(f"PC1 ({pca_proj.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca_proj.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("Colored by Projected Subtype")
axes[1].legend(fontsize=8)

plt.suptitle(
    f"Cross-Cohort Projection: GSE28521 (ASD) → GSE80655 DLPFC (SCZ+CTL)\n"
    f"ASD pathway space — {proj_ari_note}",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig(os.path.join(CROSS_DISEASE_DIR, "cross_cohort_projection_pca.png"), dpi=150, bbox_inches="tight")
plt.show()

# Save projection results
target_meta.to_csv(os.path.join(CROSS_DISEASE_DIR, "gse80655_dlpfc_projected_subtypes.csv"))
print(f"\nSaved: cross_cohort_projection_pca.png")
print(f"Saved: gse80655_dlpfc_projected_subtypes.csv")

## 15. Comprehensive Summary & Export

Aggregate all results from sections 1-14 into a single summary.
Update `results_summary.json` with cross-cohort projection metrics.

In [ ]:
# ── 15. Final Summary + Export ────────────────────────────────────────────────

# ── Compute profile correlations for JSON ─────────────────────────────────
profile_correlations = {}
if proj_has_both:
    for sub_id in sorted(disc_profiles.index):
        if sub_id in target_profiles.index:
            rho, pval = spearmanr(disc_profiles.loc[sub_id], target_profiles.loc[sub_id])
            profile_correlations[str(sub_id)] = {
                "spearman_rho": float(rho), "p_value": float(pval)
            }
else:
    present_id = list(target_profiles.index)[0]
    for disc_id in sorted(disc_profiles.index):
        rho, pval = spearmanr(disc_profiles.loc[disc_id], target_profiles.loc[present_id])
        profile_correlations[f"disc_{disc_id}_vs_tgt_{present_id}"] = {
            "spearman_rho": float(rho), "p_value": float(pval)
        }

# ── Update results_summary.json ────────────────────────────────────────────
summary_path = os.path.join(OUTPUT_DIR, "results_summary.json")
with open(summary_path) as f:
    results_summary = json.load(f)

results_summary["cross_cohort_projection"] = {
    "discovery_dataset": "GSE28521",
    "discovery_citation": "Voineagu et al. 2011, Nature",
    "discovery_platform": "Illumina HumanRef-8 v3.0 (microarray)",
    "discovery_region": "Frontal_Cortex",
    "discovery_n_samples": int(len(disc_asd_scores)),
    "target_dataset": "GSE80655",
    "target_citation": "Ramaker et al. 2017, Genome Medicine",
    "target_platform": "Illumina HiSeq 2500 (RNA-seq)",
    "target_region": "DLPFC",
    "target_n_samples": int(len(target_asd_scores)),
    "pathway_set": "ASD (autism_pathways.gmt)",
    "n_common_pathways": len(common_asd_pw),
    "discovery_silhouette": float(disc_asd_clustering.silhouette),
    "discovery_self_ari": float(disc_self_ari),
    "projected_both_subtypes": proj_has_both,
    "projected_subtype_sizes": {
        str(s): int((target_predicted == s).sum())
        for s in sorted(set(target_predicted))
    },
    "profile_correlations": profile_correlations,
}

with open(summary_path, "w") as f:
    json.dump(results_summary, f, indent=2)

# ── Print comprehensive summary ────────────────────────────────────────────
print("=" * 70)
print("NOTEBOOK 12: COMPREHENSIVE ANALYSIS SUMMARY")
print("GSE80655 — Cross-Disease Validation (SCZ / BD / MDD / Control)")
print("=" * 70)

print(f"\n--- Dataset ---")
print(f"  GEO accession: GSE80655")
print(f"  Citation: Ramaker et al. 2017, Genome Medicine")
print(f"  Total samples: {len(metadata)}")
print(f"  Diagnoses: SCZ={int((metadata['diagnosis']=='SCZ').sum())}, "
      f"BD={int((metadata['diagnosis']=='BD').sum())}, "
      f"MDD={int((metadata['diagnosis']=='MDD').sum())}, "
      f"Control={int((metadata['diagnosis']=='Control').sum())}")
print(f"  Brain regions: {sorted(metadata['brain_region'].unique())}")
print(f"  Genes: {gene_expression.shape[1]}")

print(f"\n--- Pathway Scoring ---")
print(f"  SCZ pathways: {pathway_scores_scz.shape[1]}")
print(f"  ASD pathways: {pathway_scores_asd.shape[1]}")
print(f"  Shared pathways: {len(shared_pathway_names)}")

print(f"\n--- SCZ+CTL Subtyping (SCZ pathways) ---")
print(f"  n = {len(scz_ctl_scores)} (SCZ+CTL)")
print(f"  Optimal k = {optimal_k}")
print(f"  Silhouette = {clustering.silhouette:.4f}")
n_gates = sum(1 for g in val_result.results if g.passed)
print(f"  Validation: {n_gates}/{len(val_result.results)} gates passed")
print(f"  Benchmark winner: {bench_result.best_method}")

print(f"\n--- DLPFC Region Analysis ---")
print(f"  Best DLPFC k = {best_dlpfc_k}")
print(f"  DLPFC silhouette = {dlpfc_all_results[best_dlpfc_k]['silhouette']:.4f}")

print(f"\n--- Cross-Disease (ASD pathways on SCZ+CTL) ---")
print(f"  ASD-pathway k = {asd_optimal_k}")
print(f"  ASD-pathway silhouette = {asd_clustering.silhouette:.4f}")
print(f"  Cross-pathway ARI (SCZ vs ASD) = {cross_ari:.4f}")
mean_corr = float(np.mean(list(shared_corr.values()))) if shared_corr else 0.0
print(f"  Mean shared pathway correlation = {mean_corr:.4f}")

print(f"\n--- Multi-Diagnosis Pooled Clustering ---")
print(f"  n = {int(len(all_meta))} (all diagnoses)")
print(f"  Optimal k = {pool_optimal_k}")
print(f"  Silhouette = {pool_clustering.silhouette:.4f}")
print(f"  Chi-squared (subtype~diagnosis): p = {p_value:.2e}")
print(f"  Chi-squared (subtype~region): p = {p_region:.2e}")

print(f"\n--- Cross-Cohort Projection (GSE28521 → GSE80655) ---")
print(f"  Pathway set: ASD pathways")
print(f"  Discovery: GSE28521 FC ({len(disc_asd_scores)} samples)")
print(f"  Target: GSE80655 DLPFC SCZ+CTL ({len(target_asd_scores)} samples)")
print(f"  Common pathways: {len(common_asd_pw)}")
print(f"  Discovery self-ARI: {disc_self_ari:.4f}")
print(f"  Both subtypes projected: {proj_has_both}")
if profile_correlations:
    for key, vals in profile_correlations.items():
        print(f"  Profile corr ({key}): rho={vals['spearman_rho']:.4f}, p={vals['p_value']:.2e}")

print(f"\n--- Output Files ---")
print(f"  {OUTPUT_DIR}/")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "    " + "  " * level
    subdir = os.path.basename(root)
    if level > 0:
        print(f"{indent}{subdir}/")
    for file in sorted(files):
        print(f"{indent}  {file}")

print(f"\nFramework version: 0.3.0")
print(f"Seed: {SEED}")
print(f"\n{'=' * 70}")
print("NOTEBOOK 12 COMPLETE")
print(f"{'=' * 70}")

---

## References

1. Ramaker RC, et al. (2017). Post-mortem molecular profiling of three psychiatric
   disorders. *Genome Medicine*, 9:72.
   [PMID: 28754123](https://pubmed.ncbi.nlm.nih.gov/28754123/)

2. Voineagu I, et al. (2011). Transcriptomic analysis of autistic brain reveals
   convergent molecular pathology. *Nature*, 474(7351):380-384.
   [PMID: 21614001](https://pubmed.ncbi.nlm.nih.gov/21614001/)

3. Chauhan R (2026). Pathway Subtyping Framework v0.3.0. *Zenodo*.
   [DOI: 10.5281/zenodo.18442426](https://doi.org/10.5281/zenodo.18442426)

## Data Availability

- **GSE80655:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE80655
- **GSE28521:** https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE28521
- **Framework:** https://github.com/topmist-admin/pathway-subtyping-framework
- **PyPI:** `pip install pathway-subtyping`

## License

This notebook is released under CC-BY 4.0.